# Mars Compression 03 — SegFormer baseline (20 epochs)

This notebook adds **new baseline code** to Backup 10 without changing any validated module or running another LR pilot. It is **not** Backup 11.

**Before running:** The Kaggle GPU runtime must have Backup 10 restored at `/kaggle/working/mars_work` and the unified dataset at `/tmp/mars_data`. If the runtime has reset, use the existing pilot notebook's restore and dataset-staging cells; do not rerun the full audit without reason.

**Step 1:** Run the prerequisite check. **Step 2:** Write the runner. **Step 3:** Run a non-training preflight. Create and verify Backup 11 only after review, then start Stage 1. The locked PRD and roadmap were not included in the received files and should be reviewed before training.


In [1]:
from pathlib import Path
import shutil, json, hashlib

SRC = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/backup_10_lr_selected")
DST = Path("/kaggle/working/mars_work")

# Restore Backup 10
DST.mkdir(parents=True, exist_ok=True)

for item in SRC.iterdir():
    target = DST / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Verify all files against Backup 10 manifest
manifest = json.loads((DST / "MANIFEST.json").read_text())
bad = []

for rel, info in manifest["files"].items():
    p = DST / rel
    if not p.is_file():
        bad.append(f"MISSING: {rel}")
    else:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        if h != info["sha256"]:
            bad.append(f"HASH MISMATCH: {rel}")

if bad:
    print("RESTORE FAILED")
    print("\n".join(bad))
else:
    print("RESTORE SUCCESSFUL")
    print("Verified files:", len(manifest["files"]))
    print("Selected LR:", json.loads(
        (DST / "results/lr_pilot/lr_selection.json").read_text()
    )["selected_lr"])
    print("Project:", DST)

RESTORE SUCCESSFUL
Verified files: 44
Selected LR: 0.0001
Project: /kaggle/working/mars_work


In [2]:

import json
import shutil
import time
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


# ============================================================
# 0. PATHS AND SETUP
# ============================================================

R = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
VT = R / "dataset_v3_512_val_test_csv" / "dataset_v3_512"

L = Path("/tmp/mars_data")
W = Path("/kaggle/working/mars_work")

assert R.exists(), f"Dataset not found: {R}"

for sub in [
    "checkpoints/segformer_v1",
    "code",
    "configs",
    "figures",
    "logs",
    "splits",
    "results/lr_pilot",
    "results/baseline",
    "results/compression",
    "backups"
]:
    (W / sub).mkdir(parents=True, exist_ok=True)

src_dirs = {
    "train": [
        R / f"dataset_v3_512_train_part{i}" / "dataset_v3_512" / "train"
        for i in (1, 2, 3)
    ],
    "val": [VT / "val"],
    "test_msl": [VT / "test_msl"],
    "test_mer": [VT / "test_mer"]
}

expected = {
    "train": 9803,
    "val": 1192,
    "test_msl": 311,
    "test_mer": 195
}

VALID = {0, 1, 2, 3, 255}


# ============================================================
# 1. BUILD COPY LIST
# ============================================================

tasks = []

for split, parts in src_dirs.items():
    for p in parts:
        for kind in ("img", "mask"):
            source_dir = p / kind
            assert source_dir.exists(), f"Missing directory: {source_dir}"

            for f in source_dir.glob("*.png"):
                tasks.append((f, L / split / kind / f.name))

for name in expected:
    tasks.append((VT / f"{name}.csv", L / f"{name}.csv"))

tasks.append((VT / "preprocessing.json", L / "preprocessing.json"))
tasks.append((R / "split_v3_FROZEN.json", L / "split_v3_FROZEN.json"))

print("Files to place:", len(tasks), flush=True)

# Confirm all sources exist.
missing_sources = [str(s) for s, d in tasks if not s.is_file()]
assert not missing_sources, f"Missing source files: {missing_sources[:10]}"

# Prevent filename collisions when merging train parts.
dsts = [d for s, d in tasks]
assert len(set(dsts)) == len(dsts), (
    "FILENAME COLLISION between training parts. Stop."
)

# Confirm expected file count.
expected_files = 2 * sum(expected.values()) + 6
assert len(tasks) == expected_files, (
    f"Unexpected task count: {len(tasks)} vs {expected_files}"
)


# ============================================================
# 2. THREADED COPY TO LOCAL TEMPORARY STORAGE
# ============================================================

def copy_one(task):
    source, destination = task

    if (
        destination.exists()
        and destination.stat().st_size == source.stat().st_size
    ):
        return "skipped"

    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary = destination.with_suffix(
        destination.suffix + ".part"
    )

    shutil.copyfile(source, temporary)
    temporary.replace(destination)

    if destination.stat().st_size != source.stat().st_size:
        return "SIZE_MISMATCH"

    return "copied"


print("\n=== COPYING DATASET ===", flush=True)

t0 = time.time()

with ThreadPoolExecutor(max_workers=16) as executor:
    copy_results = list(
        tqdm(
            executor.map(copy_one, tasks),
            total=len(tasks),
            desc="Copying files",
            mininterval=2
        )
    )

print(
    "Copy results:",
    dict(Counter(copy_results)),
    f"| Time: {time.time() - t0:.1f}s",
    flush=True
)

assert "SIZE_MISMATCH" not in copy_results, (
    "File size mismatch after copying."
)


# ============================================================
# 3. SHA256 VERIFICATION OF EVERY FILE
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def verify_copy(task):
    source, destination = task

    source_hash = sha256_file(source)
    destination_hash = sha256_file(destination)

    return (
        str(destination.relative_to(L)),
        source_hash,
        destination_hash
    )


print("\n=== SHA256 VERIFICATION ===", flush=True)

with ThreadPoolExecutor(max_workers=16) as executor:
    verification = list(
        tqdm(
            executor.map(verify_copy, tasks),
            total=len(tasks),
            desc="SHA256",
            mininterval=2
        )
    )

mismatches = [
    path for path, source_hash, dest_hash in verification
    if source_hash != dest_hash
]

print("SHA256 mismatches:", len(mismatches), mismatches[:5])

assert not mismatches, "SHA256 verification failed. Stop."

manifest = {
    path: source_hash
    for path, source_hash, destination_hash in verification
}

with open(W / "logs" / "dataset_sha256_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

shutil.copyfile(
    L / "split_v3_FROZEN.json",
    W / "splits" / "split_v3_FROZEN.json"
)

print("SHA256 manifest saved.", flush=True)


# ============================================================
# 4. LOCAL IMAGE AND MASK AUDIT
# ============================================================

def check_image(path):
    with Image.open(path) as im:
        return path.stem, im.size, im.mode


def check_mask(path):
    with Image.open(path) as mask:
        array = np.asarray(mask, dtype=np.uint8)

        return (
            path.stem,
            mask.size,
            mask.mode,
            np.bincount(array.ravel(), minlength=256)
        )


all_ok = True
audit = {}
stems_by_split = {}

print("\n=== IMAGE / MASK AUDIT ===", flush=True)

for split in expected:

    images = sorted((L / split / "img").glob("*.png"))
    masks = sorted((L / split / "mask").glob("*.png"))

    with ThreadPoolExecutor(max_workers=16) as executor:

        image_results = list(
            tqdm(
                executor.map(check_image, images),
                total=len(images),
                desc=f"{split} images",
                mininterval=2
            )
        )

        mask_results = list(
            tqdm(
                executor.map(check_mask, masks),
                total=len(masks),
                desc=f"{split} masks",
                mininterval=2
            )
        )

    image_stems = [r[0] for r in image_results]
    mask_stems = [r[0] for r in mask_results]

    image_set = set(image_stems)
    mask_set = set(mask_stems)

    duplicate_images = len(image_stems) - len(image_set)
    duplicate_masks = len(mask_stems) - len(mask_set)

    unpaired_images = image_set - mask_set
    unpaired_masks = mask_set - image_set

    bad_images = [
        r for r in image_results
        if r[1] != (512, 512) or r[2] != "L"
    ]

    bad_masks = [
        r[:3] for r in mask_results
        if r[1] != (512, 512) or r[2] not in ("L", "P")
    ]

    counts = np.sum(
        [r[3] for r in mask_results],
        axis=0
    )

    present = {
        int(v) for v in np.nonzero(counts)[0]
    }

    unexpected = sorted(present - VALID)
    missing_classes = sorted(VALID - present)

    ok = (
        len(images) == expected[split]
        and len(masks) == expected[split]
        and duplicate_images == 0
        and duplicate_masks == 0
        and not unpaired_images
        and not unpaired_masks
        and not bad_images
        and not bad_masks
        and not unexpected
    )

    all_ok &= ok
    stems_by_split[split] = image_set

    labelled_pixels = int(counts[:4].sum())

    class_shares = {}

    if labelled_pixels > 0:
        for i, name in enumerate(
            ["soil", "bedrock", "sand", "bigrock"]
        ):
            class_shares[name] = round(
                100 * counts[i] / labelled_pixels, 2
            )

    ignore_share = (
        100 * counts[255] / counts.sum()
        if counts.sum() else 0
    )

    print(
        f"\n[{split}] "
        f"images={len(images)} masks={len(masks)} "
        f"expected={expected[split]}"
    )

    print(
        f"Duplicates: images={duplicate_images}, masks={duplicate_masks}"
    )

    print(
        f"Unpaired: images={len(unpaired_images)}, "
        f"masks={len(unpaired_masks)}"
    )

    print(
        f"Bad formats: images={len(bad_images)}, "
        f"masks={len(bad_masks)}"
    )

    print(
        f"Unexpected labels={unexpected}, "
        f"classes absent={missing_classes}"
    )

    print(
        f"Ignore pixels={ignore_share:.2f}% | "
        f"Class shares={class_shares}"
    )

    print("STATUS:", "OK" if ok else "PROBLEM", flush=True)

    audit[split] = {
        "images": len(images),
        "masks": len(masks),
        "ok": bool(ok),
        "label_pixel_counts": {
            int(k): int(counts[k]) for k in present
        }
    }


# ============================================================
# 5. CHECK SPLIT OVERLAP
# ============================================================

print("\n=== SPLIT OVERLAP ===")

names = list(stems_by_split)

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        overlap = (
            stems_by_split[names[i]]
            & stems_by_split[names[j]]
        )

        print(
            f"{names[i]} & {names[j]}: {len(overlap)}"
        )

        all_ok &= (len(overlap) == 0)


# ============================================================
# 6. CSV VALIDATION
# ============================================================

def to_stem(value):
    return Path(str(value).replace("\\", "/")).stem


print("\n=== CSV VALIDATION ===")

csv_columns = {}

for name in expected:

    df = pd.read_csv(L / f"{name}.csv")

    print(
        f"\n{name}.csv rows={len(df)} "
        f"expected={expected[name]}"
    )

    print("Columns:", list(df.columns))

    matching_columns = []

    for column in df.columns:

        if df[column].dtype == object:

            stems = df[column].map(to_stem)

            if (
                stems.is_unique
                and set(stems) == stems_by_split[name]
            ):
                matching_columns.append(column)

    csv_ok = (
        len(df) == expected[name]
        and bool(matching_columns)
    )

    print(
        "Exact filename-matching columns:",
        matching_columns,
        "->",
        "OK" if csv_ok else "PROBLEM"
    )

    csv_columns[name] = matching_columns
    all_ok &= csv_ok

    if name == "train":
        print("\nTrain CSV sample:")
        print(df.head(2).to_string())


# ============================================================
# 7. FROZEN SPLIT JSON STRUCTURE VALIDATION
# ============================================================

print("\n=== FROZEN SPLIT JSON VALIDATION ===")

with open(L / "split_v3_FROZEN.json") as f:
    split_json = json.load(f)

candidates = []


def walk_json(obj, path="$"):

    if isinstance(obj, list):

        if obj and all(isinstance(x, str) for x in obj):

            candidates.append((
                path,
                {to_stem(x) for x in obj},
                len(obj)
            ))

        else:

            for index, item in enumerate(obj[:50]):
                walk_json(item, f"{path}[{index}]")

    elif isinstance(obj, dict):

        if (
            len(obj) > 50
            and all(isinstance(k, str) for k in obj)
        ):
            candidates.append((
                path + " (dict keys)",
                {to_stem(k) for k in obj},
                len(obj)
            ))

        for key, value in obj.items():
            walk_json(value, f"{path}.{key}")


walk_json(split_json)

json_ok = True

for name in expected:

    hits = [
        path
        for path, stem_set, count in candidates
        if (
            stem_set == stems_by_split[name]
            and count == expected[name]
        )
    ]

    print(
        f"{name}: exact filename match = "
        f"{hits if hits else 'NOTHING'}"
    )

    json_ok &= bool(hits)


if json_ok:

    print("Frozen split: VERIFIED against image filenames")

else:

    print(
        "Frozen split: UNVERIFIED. "
        "JSON may store indices, metadata or other identifiers."
    )

    print("\nJSON structure:")

    def describe(obj, depth=0):

        if isinstance(obj, dict):

            for key, value in list(obj.items())[:15]:

                description = (
                    f"len={len(value)}"
                    if isinstance(value, (list, dict))
                    else repr(value)[:70]
                )

                print(
                    "  " * depth,
                    f"{key}: {type(value).__name__} {description}"
                )

                if isinstance(value, dict) and depth < 2:
                    describe(value, depth + 1)

    describe(split_json)


# ============================================================
# 8. SAVE AUDIT AND FINAL STATUS
# ============================================================

audit["all_data_csv_checks_passed"] = bool(all_ok)
audit["frozen_split_verified"] = bool(json_ok)

with open(W / "logs" / "dataset_audit_kaggle.json", "w") as f:
    json.dump(audit, f, indent=2)

print("\n" + "=" * 60)

print(
    "DATA + CSV AUDIT:",
    "ALL CHECKS PASSED" if all_ok else "FAILED"
)

print(
    "FROZEN SPLIT JSON:",
    "VERIFIED" if json_ok else "UNVERIFIED - INVESTIGATE"
)

print(
    "OVERALL:",
    "READY FOR PIPELINE VALIDATION"
    if all_ok and json_ok
    else "DO NOT PROCEED UNTIL ISSUES ARE RESOLVED"
)

print("\nSaved files:")
print(W / "logs" / "dataset_audit_kaggle.json")
print(W / "logs" / "dataset_sha256_manifest.json")
print(W / "splits" / "split_v3_FROZEN.json")

print("\nLocal dataset:", L)
print("Audit complete.")


Files to place: 23008

=== COPYING DATASET ===


Copying files:   0%|          | 0/23008 [00:00<?, ?it/s]

Copy results: {'copied': 23008} | Time: 87.2s

=== SHA256 VERIFICATION ===


SHA256:   0%|          | 0/23008 [00:00<?, ?it/s]

SHA256 mismatches: 0 []
SHA256 manifest saved.

=== IMAGE / MASK AUDIT ===


train images:   0%|          | 0/9803 [00:00<?, ?it/s]

... (7 outputs left out)


[test_msl] images=311 masks=311 expected=311
Duplicates: images=0, masks=0
Unpaired: images=0, masks=0
Bad formats: images=0, masks=0
Unexpected labels=[], classes absent=[]
Ignore pixels=40.60% | Class shares={'soil': np.float64(45.8), 'bedrock': np.float64(30.91), 'sand': np.float64(23.1), 'bigrock': np.float64(0.2)}
STATUS: OK


test_mer images:   0%|          | 0/195 [00:00<?, ?it/s]

test_mer masks:   0%|          | 0/195 [00:00<?, ?it/s]


[test_mer] images=195 masks=195 expected=195
Duplicates: images=0, masks=0
Unpaired: images=0, masks=0
Bad formats: images=0, masks=0
Unexpected labels=[], classes absent=[]
Ignore pixels=37.17% | Class shares={'soil': np.float64(49.01), 'bedrock': np.float64(23.21), 'sand': np.float64(26.78), 'bigrock': np.float64(1.01)}
STATUS: OK

=== SPLIT OVERLAP ===
train & val: 0
train & test_msl: 0
train & test_mer: 0
val & test_msl: 0
val & test_mer: 0
test_msl & test_mer: 0

=== CSV VALIDATION ===

train.csv rows=9803 expected=9803
Columns: ['image_path', 'label_path', 'labeled_frac', 'sclk', 'F', 'class_0', 'class_1', 'class_2', 'class_3', 'img_512', 'mask_512']
Exact filename-matching columns: ['image_path', 'label_path', 'img_512', 'mask_512'] -> OK

Train CSV sample:
                                                     image_path                                                      label_path  labeled_frac         sclk     F   class_0  class_1  class_2  class_3                           

In [4]:
import json
import hashlib
from pathlib import Path

W = Path("/kaggle/working/mars_work")
L = Path("/tmp/mars_data")

split_info = json.loads(
    (W / "splits/split_v3_FROZEN.json").read_text()
)["files"]

for split, info in split_info.items():
    csv_path = L / f"{split}.csv"

    actual_md5 = hashlib.md5(csv_path.read_bytes()).hexdigest()
    expected_md5 = info["md5"]

    print(
        split,
        "MATCH" if actual_md5 == expected_md5 else "NO MATCH",
        "| Expected:", expected_md5,
        "| Actual:", actual_md5
    )

train NO MATCH | Expected: 8d02c9c95be90e4a89663a84c3fd82b7 | Actual: 1e72ce98a07a23c47697e220518e9c36
val NO MATCH | Expected: 0a86468c938bf3009b6db117c881c38d | Actual: 54a263a86d0c962e513152b6c830d302
test_msl NO MATCH | Expected: 46a63b56c865a1d2c6c8bc13a90bdeaa | Actual: c8417796db62007412fca69021f3059b
test_mer NO MATCH | Expected: 289521ca2a432d9cb63c6fb8138a5f41 | Actual: fc09d5b09b751fcc2fecbb2d0da3abc4


In [15]:
from pathlib import Path
import torch
R = Path("/kaggle/working/mars_work")
D = Path("/tmp/mars_data")
required = [R / "code/mars_common.py", R / "code/mars_det.py", R / "code/ckpt_utils.py", R / "code/tracking.py", R / "checkpoints/lr_pilot/init_seed42.pt", D / "train.csv", D / "val.csv"]
missing = [str(p) for p in required if not p.is_file()]
print("CUDA available:", torch.cuda.is_available())
print("Missing prerequisites:", missing or "None")
assert torch.cuda.is_available() and not missing, "Restore Backup 10 or assemble the dataset before continuing"


CUDA available: True
Missing prerequisites: None


In [6]:
from pathlib import Path

W = Path("/kaggle/working/mars_work")
D = Path("/tmp/mars_data")

print("Backup 10 restored:", (W / "code/tracking.py").exists())
print("Dataset available:", (D / "train.csv").exists())
print("Initial checkpoint:", (W / "checkpoints/lr_pilot/init_seed42.pt").exists())

Backup 10 restored: True
Dataset available: True
Initial checkpoint: True


In [20]:
import sys, os, subprocess, importlib.util
if importlib.util.find_spec("wandb") is None:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
        print("wandb installed")
    except Exception as e:
        print("wandb installation unavailable, local-only logging will still work:", type(e).__name__)
else:
    print("wandb package already present")

secret_available = bool(os.getenv("WANDB_API_KEY"))
if not secret_available:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_available = bool(UserSecretsClient().get_secret("WANDB_API_KEY"))
    except Exception:
        pass
print("W&B API key available securely:", secret_available)
print("W&B mode:", os.getenv("WANDB_MODE", "online"))
print("No credential value was displayed or saved in the notebook.")

wandb package already present
W&B API key available securely: True
W&B mode: online
No credential value was displayed or saved in the notebook.


In [21]:
try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    key = client.get_secret("WANDB_API_KEY")

    if key:
        import os
        os.environ["WANDB_API_KEY"] = key
        print("SUCCESS: W&B key retrieved and made available.")
    else:
        print("Secret returned an empty value.")

except Exception as e:
    print("Secret retrieval failed:", type(e).__name__)

SUCCESS: W&B key retrieved and made available.


In [26]:
%%writefile /kaggle/working/mars_work/code/baseline_train.py
"""Approved 20-epoch Mars SegFormer baseline, compatible with Backup 10.

Run in Kaggle after restoring Backup 10 and /tmp/mars_data:
  python /kaggle/working/mars_work/code/baseline_train.py --stage 1
  # after Backup 12 is created/restored at exactly iteration 12250
  python /kaggle/working/mars_work/code/baseline_train.py --stage 2

Stage 1 trains to iteration 12250; stage 2 to 24500.  Never reads test splits.
No validated module from Backup 10 is modified. W&B is optional; local data is authoritative.
"""
import argparse
import csv
import json
import math
import os
import re
import shutil
import sys
import time
import importlib.util
import secrets
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import ckpt_utils as cu
import mars_common as mc
import mars_det as md
import tracking as tk

TOTAL = 24500
STAGE1_END = 12250
SPE = 1225
CN = ["soil", "bedrock", "sand", "bigrock"]


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf8") as f:
        json.dump(obj, f, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def truncate_csv(path, last_iteration):
    """Retain the header and only committed rows. Reconcile a torn last row."""
    path = Path(path)
    if not path.exists():
        return 0
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        if not columns or "iteration" not in columns:
            raise ValueError(f"Unexpected CSV format: {path}")
        rows = list(reader)
    kept = []
    for row in rows:
        try:
            it = int(float(row["iteration"]))
            valid = all(row.get(col) is not None for col in columns)
        except (TypeError, ValueError):
            valid = False
            it = last_iteration + 1
        if valid and it <= last_iteration:
            kept.append(row)
    tmp = path.with_suffix(".csv.tmp")
    with tmp.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(kept)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)
    return len(rows) - len(kept)


def same_file(src, dst):
    try:
        return os.path.samefile(src, dst)
    except OSError:
        return False


def update_best_pointer(ck_dir, best_file, iteration, miou4, val_loss, epoch):
    """Best checkpoints are immutable; best.pt and JSON pointer swap atomically."""
    dest = ck_dir / best_file
    if not dest.is_file():
        raise FileNotFoundError(f"Best checkpoint missing: {dest}")
    tmp = ck_dir / "best.pt.tmp"
    tmp.unlink(missing_ok=True)
    # Kaggle is Linux: hardlink avoids copying large checkpoints.
    os.link(dest, tmp)
    os.replace(tmp, ck_dir / "best.pt")
    atomic_json(ck_dir / "best_pointer.json", dict(filename=best_file,
                                                     iteration=iteration,
                                                     epoch=epoch,
                                                     miou4=miou4,
                                                     validation_loss=val_loss))


def recover_best_files(ck_dir, it0, extra, best, best_loss):
    """Quarantine checkpoints from a failed post-last.pt epoch; never delete them."""
    orphan_dir = ck_dir / "orphans"
    moved = []
    for path in sorted(ck_dir.glob("best_epoch*_it*.pt")):
        match = re.fullmatch(r"best_epoch\d+_it(\d+)\.pt", path.name)
        if match and int(match.group(1)) > it0:
            orphan_dir.mkdir(parents=True, exist_ok=True)
            dest = orphan_dir / path.name
            if dest.exists():
                raise FileExistsError(f"Orphan quarantine collision: {dest}")
            os.replace(path, dest)
            moved.append(path.name)
    name = extra.get("best_file")
    if name:
        if not (ck_dir / name).is_file():
            raise FileNotFoundError(f"Committed best checkpoint is missing: {name}")
        update_best_pointer(ck_dir, name, extra["best_iteration"], best,
                            best_loss, extra["best_epoch"])
    else:
        # No best model is committed as of the resumed last.pt.
        for f in (ck_dir / "best.pt", ck_dir / "best_pointer.json"):
            if f.exists():
                orphan_dir.mkdir(parents=True, exist_ok=True)
                dest = orphan_dir / (f.name + f".after_it{it0}")
                if dest.exists():
                    raise FileExistsError(f"Orphan quarantine collision: {dest}")
                os.replace(f, dest)
    return moved


def compare_best(miou, loss, ep, best, best_loss, best_epoch):
    """Highest mIoU4; exact tie -> lower loss; still tied -> earlier epoch."""
    if not math.isfinite(miou) or not math.isfinite(loss):
        raise ValueError("Non-finite validation metric; training stopped")
    if miou > best:
        return True
    if miou == best and loss < best_loss:
        return True
    return miou == best and loss == best_loss and ep < best_epoch


def build_config():
    return dict(
        stage="segformer_baseline_v1", model="nvidia/mit-b0", seed=42,
        batch_size=8, steps_per_epoch=SPE, epochs=20, total_iters=TOTAL,
        start_lr=1e-4, optimizer="AdamW", weight_decay=0.01,
        warmup=100, final_frac=0.05, amp="fp16", ignore_index=255,
        workers=2, val_batch_size=16, train_log_every=25,
        checkpoint_every=500, train_split="train", val_split="val",
        augmentation="mars_det deterministic horizontal flip",
        no_class_weights=True, no_grad_clip=True, early_stopping=False,
        cudnn_deterministic=True, cudnn_benchmark=False,
        init_state_hash="b16b52beb330a9e331d1f16e58b0d55d81070286b28323f7610f88ec0fb20f69",
        data_stream_first_4900_hash="9327320a886992000712839c7831c8346159f49c09b617e8984ec6a75222caaa",
    )


def preflight(root, cfg):
    if not torch.cuda.is_available():
        raise RuntimeError("Kaggle CUDA GPU is required for the approved FP16 training")
    if not (root / "checkpoints" / "lr_pilot" / "init_seed42.pt").is_file():
        raise FileNotFoundError("Backup 10 is not restored: missing init_seed42.pt")
    required = ["train.csv", "val.csv"]
    for rel in required:
        if not (Path(mc.DATA) / rel).is_file():
            raise FileNotFoundError(f"Dataset not assembled: {Path(mc.DATA) / rel}")
    n_train = len(mc.MarsSeg("train"))
    n_val = len(mc.MarsSeg("val"))
    if (n_train, n_val) != (9803, 1192):
        raise ValueError(f"Frozen split counts changed: {(n_train, n_val)}")
    if n_train // cfg["batch_size"] != SPE:
        raise ValueError("Expected 1225 full batches per epoch")
    if cfg["total_iters"] != cfg["epochs"] * SPE or STAGE1_END != 10 * SPE:
        raise ValueError("Approved epoch/iteration schedule changed")
    if not (root / "splits" / "split_v3_FROZEN.json").is_file():
        raise FileNotFoundError("Frozen split descriptor missing")
    sp = json.loads((root / "splits" / "split_v3_FROZEN.json").read_text())
    assert sp["status"] == "FROZEN"
    assert sp["files"]["train"]["rows"] == n_train
    assert sp["files"]["val"]["rows"] == n_val
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return n_train, n_val



def prepare_wandb(res, cfg, it0):
    """Securely configure optional W&B; one Tracker owns one W&B run.

    Keeps the unchanged tracking.py as the only telemetry implementation.
    A resumed run reuses the prior W&B ID only if its last submitted step
    is not ahead of the committed checkpoint. On rewind, create a new
    grouped W&B segment because W&B history cannot be rolled back.
    """
    if os.getenv("MARS_USE_WANDB", "1").strip().lower() in ("0", "false", "no", "off"):
        print("W&B explicitly disabled; local logging remains enabled")
        return False, None
    if importlib.util.find_spec("wandb") is None:
        print("W&B package unavailable; local logging remains enabled")
        return False, None
    mode = os.getenv("WANDB_MODE", "online").strip().lower()
    if mode not in ("offline", "disabled", "dryrun") and not os.getenv("WANDB_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
            if key:
                os.environ["WANDB_API_KEY"] = key
        except Exception:
            pass  # tracking.py also accepts Kaggle Secrets; never print the key.
    if mode not in ("offline", "dryrun") and not os.getenv("WANDB_API_KEY"):
        print("No W&B API key available from Kaggle Secrets/environment; using local CSV only")
        return False, None
    if mode == "disabled":
        return False, None
    os.environ.setdefault("WANDB_INIT_TIMEOUT", "30")
    os.environ.setdefault("WANDB_DIR", str(res / "wandb_runs"))
    Path(os.environ["WANDB_DIR"]).mkdir(parents=True, exist_ok=True)
    state_path = res / "wandb_run.json"
    state = json.loads(state_path.read_text()) if state_path.is_file() else None
    if state is not None and state.get("config_hash") != cu.config_hash(cfg):
        raise ValueError("W&B run metadata has a different locked configuration")
    if state is None or int(state.get("last_submitted_iteration", -1)) > it0:
        previous = state.get("run_id") if state else None
        state = dict(run_id=secrets.token_hex(8), config_hash=cu.config_hash(cfg),
                     last_submitted_iteration=-1, previous_run_id=previous,
                     resume_from_iteration=it0)
        atomic_json(state_path, state)
        if previous:
            print("W&B history exceeded committed checkpoint; new grouped W&B segment for safe rewind")
    os.environ["WANDB_RUN_ID"] = state["run_id"]
    os.environ["WANDB_RESUME"] = "allow"
    return True, state


def resume_tracker_csv_fields(tracker):
    """Use the existing CSV header when appending, even at epoch boundary."""
    path = Path(f"{tracker.prefix}_log.csv")
    if path.is_file():
        with path.open(newline="") as f:
            header = next(csv.reader(f), None)
        if not header or "iteration" not in header:
            raise ValueError(f"Invalid existing local log header: {path}")
        tracker.fields["log"] = header


def json_numbers(values):
    """Represent undefined per-class values as JSON null (never NaN)."""
    return [float(v) if v is not None and math.isfinite(float(v)) else None
            for v in values]


def wb_validation_scalars(row):
    """Scalar values only. Train and validation share one W&B step per 25 iters."""
    out = {f"validation_{k}": v for k, v in row.items()
           if k not in ("iteration", "epoch") and isinstance(v, (int, float))
           and math.isfinite(float(v))}
    out["best_checkpoint_event"] = int(bool(row["is_best"]))
    return out


def run(root, stage):
    cfg = build_config()
    root = Path(root).resolve()
    n_train, n_val = preflight(root, cfg)
    end = STAGE1_END if stage == 1 else TOTAL
    ck_dir = root / "checkpoints" / "baseline"
    res = root / "results" / "baseline"
    last = ck_dir / "last.pt"
    init_file = root / "checkpoints" / "lr_pilot" / "init_seed42.pt"
    ck_dir.mkdir(parents=True, exist_ok=True)
    res.mkdir(parents=True, exist_ok=True)

    cfg_path = root / "configs" / "baseline_protocol.json"
    if cfg_path.exists():
        disk_cfg = json.loads(cfg_path.read_text())
        if disk_cfg != cfg:
            raise ValueError("Existing baseline_protocol.json does not match this locked protocol")
    else:
        atomic_json(cfg_path, cfg)

    # Exactly the validated initial model/weights loading method.
    mc.set_seed(cfg["seed"])
    model = mc.new_model()
    init = torch.load(init_file, map_location="cpu", weights_only=True)
    if init["state_hash"] != cfg["init_state_hash"]:
        raise ValueError("The initialized checkpoint state hash differs from Backup 10")
    model.load_state_dict(init["state_dict"])
    mc.set_seed(cfg["seed"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["start_lr"], weight_decay=cfg["weight_decay"])
    sched = cu.make_scheduler(opt, cfg["total_iters"], cfg["warmup"], cfg["final_frac"])
    scaler = torch.amp.GradScaler(mc.DEV, enabled=mc.USE_AMP)

    it0, best, best_loss, best_epoch = 0, -1., float("inf"), 10 ** 9
    extra = dict(skipped=0, epoch_loss_sum=0.0, epoch_seen=0,
                 best_file=None, best_iteration=None, best_epoch=None)
    if last.exists():
        ck = cu.load_checkpoint(last, model, opt, sched, scaler, cfg=cfg, restore_rng=True)
        it0, best = ck["iteration"], float(ck["best_metric"])
        extra.update(ck["extra"])
        if extra.get("best_file"):
            best_loss = float(extra["best_loss"])
            best_epoch = int(extra["best_epoch"])
        print(f"Resuming baseline from last.pt iteration {it0}; config {ck['config_hash']}")
        moved = recover_best_files(ck_dir, it0, extra, best, best_loss)
        if moved:
            print("Quarantined uncommitted best checkpoints:", moved)
        for p in (res / "train_log.csv", res / "validation_log.csv"):
            removed = truncate_csv(p, it0)
            if removed:
                print(f"Reconciled {p.name}: removed {removed} uncommitted rows")
        # A final epoch confusion matrix is only committed after last.pt.
        cm_dir = res / "confusion_matrices"
        for p in sorted(cm_dir.glob("epoch_*.json")):
            m = re.fullmatch(r"epoch_(\d+)\.json", p.name)
            if m and int(m.group(1)) * SPE > it0:
                dest_dir = cm_dir / "orphans"
                dest_dir.mkdir(parents=True, exist_ok=True)
                dest = dest_dir / p.name
                if dest.exists():
                    raise FileExistsError(f"Orphan confusion-matrix collision: {dest}")
                os.replace(p, dest)
    else:
        if stage == 2:
            raise RuntimeError("Stage 2 requires a genuine Backup 12 last.pt at iteration 12250")
        if any(ck_dir.glob("best_epoch*_it*.pt")) or (res / "train_log.csv").exists():
            raise RuntimeError("Baseline files exist without last.pt; refuse to overwrite recovery state")
        print(f"Starting approved 20-epoch baseline from shared init seed {cfg['seed']}")

    if stage == 1 and it0 > STAGE1_END:
        raise RuntimeError("Stage 1 already passed epoch 10; use --stage 2")
    if stage == 2 and it0 < STAGE1_END:
        raise RuntimeError("Stage 2 must start at or after iteration 12250 from Backup 12")
    if it0 > end:
        raise RuntimeError(f"Checkpoint iteration {it0} exceeds this stage's endpoint {end}")
    if it0 == end:
        print(f"Stage {stage} already complete at iteration {it0}; no new training performed")
        return
    if it0 != 0 and it0 % SPE == 0 and (res / "validation_log.csv").exists():
        with (res / "validation_log.csv").open() as f:
            rows = list(csv.DictReader(f))
        if not rows or int(float(rows[-1]["iteration"])) != it0:
            raise RuntimeError("Checkpoint is on epoch boundary but final validation row is missing")

    # W&B is supplementary: one run and one W&B event per training-log step.
    # Preserve torch / numpy / python / CUDA RNG across W&B init/login.
    rng_for_tracking = cu.get_rng_state()
    try:
        try:
            wb_enabled, wb_state = prepare_wandb(res, cfg, it0)
        except Exception as e:
            print(f"W&B setup unavailable ({type(e).__name__}); continuing with local CSV only")
            wb_enabled, wb_state = False, None
        train_tracker = tk.Tracker(res / "train", project="mars-terrain-segmentation",
                                   name="segformer-b0-seed42", group="baseline-segformer-b0",
                                   config=cfg, use_wandb=wb_enabled)
        val_tracker = tk.Tracker(res / "validation", use_wandb=False)
        resume_tracker_csv_fields(train_tracker)
        resume_tracker_csv_fields(val_tracker)
    finally:
        cu.set_rng_state(rng_for_tracking)
    print("W&B:", "connected" if train_tracker.run is not None else "local-only")
    val_loader = DataLoader(mc.MarsSeg("val"), batch_size=cfg["val_batch_size"],
                            shuffle=False, num_workers=cfg["workers"], pin_memory=True)
    loader = md.make_det_loader(cfg["seed"], cfg["batch_size"], it0, end,
                                workers=cfg["workers"], flip=True)
    # DataLoader iterator starts worker seeds, which would otherwise consume
    # the main process's saved Torch RNG and shift the resumed dropout stream.
    rng_at_boundary = cu.get_rng_state()
    iterator = iter(loader)
    cu.set_rng_state(rng_at_boundary)

    it = it0
    loss_sum = float(extra.get("epoch_loss_sum", 0.0))
    seen = int(extra.get("epoch_seen", 0))
    skipped = int(extra.get("skipped", 0))
    nonfinite_grad = int(extra.get("nonfinite_grad_norm", 0))
    epoch_elapsed_offset = float(extra.get("epoch_elapsed_s", 0.0))
    epoch_clock = time.monotonic()
    if seen != it % SPE:
        raise ValueError(f"Checkpoint epoch accumulator mismatch: seen={seen} it={it}")
    win_loss, win_gn = [], []
    tick = time.monotonic()

    def ck_extra():
        return dict(skipped=skipped, epoch_loss_sum=loss_sum, epoch_seen=seen,
                    best_file=extra.get("best_file"), best_loss=extra.get("best_loss"),
                    best_iteration=extra.get("best_iteration"),
                    best_epoch=extra.get("best_epoch"),
                    nonfinite_grad_norm=nonfinite_grad,
                    epoch_elapsed_s=epoch_elapsed_offset + (time.monotonic() - epoch_clock))

    def save_last():
        cu.save_checkpoint(last, model, opt, sched, scaler, it // SPE, it,
                           cfg, best, extra=ck_extra())

    def note_wb(step):
        if wb_state is not None and train_tracker.run is not None:
            wb_state["last_submitted_iteration"] = int(step)
            try:
                atomic_json(res / "wandb_run.json", wb_state)
            except OSError as e:
                print(f"W&B metadata write failed ({type(e).__name__}); local records remain authoritative")

    progress = tqdm(iterator, total=end - it0, desc=f"baseline stage {stage}")
    try:
        for x, y in progress:
            lr_used = float(opt.param_groups[0]["lr"])
            scale_before = scaler.get_scale() if scaler.is_enabled() else 1.0
            loss, gradnorm = mc.train_step(model, opt, scaler, x, y)
            scale_after = scaler.get_scale() if scaler.is_enabled() else 1.0
            # This detects actual GradScaler-overflow skips, not merely a nonfinite norm.
            skipped_this_step = bool(scaler.is_enabled() and scale_after < scale_before)
            if skipped_this_step:
                skipped += 1
            if not math.isfinite(gradnorm):
                nonfinite_grad += 1
            sched.step()  # Approved iteration-based schedule; never restarted on resume.
            it += 1
            if not math.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss at iteration {it}; checkpoint unchanged")
            loss_sum += loss
            seen += 1
            win_loss.append(loss)
            if math.isfinite(gradnorm):
                win_gn.append(gradnorm)

            if it % cfg["train_log_every"] == 0:
                train_row = dict(iteration=it, epoch=it / SPE,
                                 loss=float(np.mean(win_loss)), lr=lr_used,
                                 grad_norm=float(np.mean(win_gn)) if win_gn else "nan",
                                 skipped_steps_total=skipped,
                                 nonfinite_grad_norm_total=nonfinite_grad,
                                 elapsed_s=time.monotonic() - tick)
                win_loss, win_gn = [], []
            else:
                train_row = None

            if it % SPE == 0:
                epoch = it // SPE
                eval_start = time.monotonic()
                result = mc.evaluate(model, val_loader)
                eval_s = time.monotonic() - eval_start
                improved = compare_best(result["miou4"], result["loss"], epoch,
                                        best, best_loss, best_epoch)
                if improved:
                    best, best_loss, best_epoch = result["miou4"], result["loss"], epoch
                    name = f"best_epoch{epoch:02d}_it{it}.pt"
                    path = ck_dir / name
                    if path.exists():
                        raise FileExistsError(f"Refuse to overwrite immutable best checkpoint: {path}")
                    extra.update(best_file=name, best_loss=best_loss,
                                 best_iteration=it, best_epoch=epoch)
                    cu.save_checkpoint(path, model, opt, sched, scaler, epoch, it,
                                       cfg, best, extra=ck_extra())
                    update_best_pointer(ck_dir, name, it, best, best_loss, epoch)
                row = dict(iteration=it, epoch=epoch, val_loss=result["loss"],
                           miou4=result["miou4"], miou3=result["miou3"],
                           pixacc=result["pixacc"])
                for j, name in enumerate(CN):
                    row[f"iou_{name}"] = result["iou"][j]
                    row[f"recall_{name}"] = result["recall"][j]
                    row[f"precision_{name}"] = result["precision"][j]
                row.update(eval_s=eval_s,
                           epoch_duration_s=epoch_elapsed_offset + time.monotonic() - epoch_clock,
                           epoch_mean_train_loss=loss_sum / seen,
                           skipped_steps_total=skipped, nonfinite_grad_norm_total=nonfinite_grad,
                           is_best=improved)
                val_tracker.log("log", row, step=it)  # authoritative local validation CSV
                cm_dir = res / "confusion_matrices"
                atomic_json(cm_dir / f"epoch_{epoch:02d}.json",
                            dict(iteration=it, epoch=epoch, confusion_matrix=result["conf"],
                                 iou=json_numbers(result["iou"]),
                                 recall=json_numbers(result["recall"]),
                                 precision=json_numbers(result["precision"]), miou4=result["miou4"],
                                 miou3=result["miou3"], loss=result["loss"]))
                print(f"epoch {epoch:02d}/20 | val mIoU4={result['miou4']:.6f} | "
                      f"mIoU3={result['miou3']:.6f} | val loss={result['loss']:.6f} | "
                      f"best epoch={best_epoch}", flush=True)
                # Always log W&B exactly once at each iteration. At epoch ends,
                # combine training + validation scalars in the SAME tracker event.
                assert train_row is not None  # 1225 divisible by 25
                train_tracker.log("log", {**train_row, **wb_validation_scalars(row)}, step=it)
                note_wb(it)
                loss_sum, seen = 0.0, 0
                epoch_elapsed_offset = 0.0
                epoch_clock = time.monotonic()
                save_last()  # epoch fully committed, including validation
            else:
                if train_row is not None:
                    train_tracker.log("log", train_row, step=it)
                    note_wb(it)
                if it % cfg["checkpoint_every"] == 0:
                    save_last()
            progress.set_postfix(iter=it, loss=f"{loss:.3f}", lr=f"{lr_used:.2g}")
    finally:
        train_tracker.finish()
        val_tracker.finish()
    assert it == end
    assert seen == 0, "Stage must finish at an epoch boundary"
    print(f"STAGE {stage} COMPLETE | {it}/{TOTAL} iterations | best val mIoU4={best:.6f}")
    if stage == 1:
        print("STOP: create/verify genuine Backup 12 before running --stage 2.")
    else:
        print("STOP: freeze best checkpoint before the one-time final test evaluation.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", type=int, choices=(1, 2))
    parser.add_argument("--check", action="store_true", help="Preflight only. Does NOT train or create Backup 11")
    parser.add_argument("--root", default="/kaggle/working/mars_work")
    args = parser.parse_args()
    if args.check:
        root = Path(args.root)
        cfg = build_config()
        n_train, n_val = preflight(root, cfg)
        ini = torch.load(root / "checkpoints/lr_pilot/init_seed42.pt", map_location="cpu", weights_only=True)
        if ini.get("state_hash") != cfg["init_state_hash"]:
            raise ValueError("Initial model state hash metadata mismatches Backup 10")
        mc.set_seed(cfg["seed"])
        model = mc.new_model()
        model.load_state_dict(ini["state_dict"])
        del model
        print(f"PREFLIGHT PASSED: train={n_train}, val={n_val}, "
              f"config_hash={cu.config_hash(cfg)}, CUDA={torch.cuda.get_device_name(0)}")
        print("No training was performed; no Backup 11/12 was created.")
    elif args.stage is not None:
        run(args.root, args.stage)
    else:
        parser.error("Specify --check or --stage 1/2")


Overwriting /kaggle/working/mars_work/code/baseline_train.py


In [27]:
%%writefile /kaggle/working/mars_work/code/baseline_analysis.py
"""Reproduce SegFormer baseline figures/report from LOCAL saved CSV and JSON only.

After training:
    python code/baseline_analysis.py --root /kaggle/working/mars_work
No W&B, dataset, test split or model/checkpoint is accessed.
"""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CLASSES = ["soil", "bedrock", "sand", "bigrock"]
DISPLAY = ["Soil", "Bedrock", "Sand", "Big Rock"]


def _save(path):
    plt.tight_layout()
    plt.savefig(path, dpi=165, bbox_inches="tight")
    plt.close()


def _line(out, x, series, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, y in series:
        ax.plot(x, y, marker="o" if len(x) <= 20 else None, linewidth=1.7, label=label)
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.grid(alpha=0.25)
    if len(series) > 1:
        ax.legend()
    _save(out / filename)


def _confusion_plot(path, conf, epoch):
    conf = np.asarray(conf, dtype=np.int64)
    if conf.shape != (4, 4) or np.any(conf < 0):
        raise ValueError(f"Invalid 4x4 confusion matrix at epoch {epoch}")
    row_sums = conf.sum(axis=1, keepdims=True)
    norm = np.divide(conf, row_sums, out=np.zeros((4, 4), dtype=float), where=row_sums != 0)
    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(4), DISPLAY)
    ax.set_yticks(range(4), DISPLAY)
    ax.set(xlabel="Predicted class", ylabel="True class", title=f"Validation confusion matrix — epoch {epoch:02d}\nRow-normalised %; raw pixel count")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{norm[i,j]*100:.1f}%\n({conf[i,j]:,})",
                    ha="center", va="center", fontsize=8,
                    color="white" if norm[i,j] > 0.55 else "black")
    fig.colorbar(image, ax=ax, label="Share of labelled true-class pixels")
    _save(path)


def _float(f):
    return float(f) if pd.notna(f) else float("nan")


def _build_report(train, val):
    full = len(val) >= 20 and int(val["epoch"].max()) == 20
    start_end = (int(val["epoch"].min()), int(val["epoch"].max()))
    bestrow = val.loc[val["miou4"].idxmax()]
    latest = val.iloc[-1]
    recent = val.tail(min(5, len(val)))
    t_by_epoch = train.assign(epoch_idx=np.ceil(train["epoch"] - 1e-10).astype(int)).groupby("epoch_idx")["loss"].mean()
    first_epoch, last_epoch = start_end
    gap_points = []
    for _, r in val.iterrows():
        ep = int(r["epoch"])
        if ep in t_by_epoch.index:
            gap_points.append((ep, float(r["val_loss"] - t_by_epoch.loc[ep])))
    d_miou = float(latest["miou4"] - val.iloc[0]["miou4"])
    recent_range = float(recent["miou4"].max() - recent["miou4"].min())
    recent_change = float(recent.iloc[-1]["miou4"] - recent.iloc[0]["miou4"])
    d_train = float(t_by_epoch.loc[last_epoch] - t_by_epoch.loc[first_epoch]) if first_epoch in t_by_epoch.index and last_epoch in t_by_epoch.index else float("nan")
    d_val = float(latest["val_loss"] - val.iloc[0]["val_loss"])
    skip_first = int(train["skipped_steps_total"].iloc[0]) if "skipped_steps_total" in train else 0
    skipped = int(train["skipped_steps_total"].max()) if "skipped_steps_total" in train else 0
    nfg = int(train["nonfinite_grad_norm_total"].max()) if "nonfinite_grad_norm_total" in train else None
    lr_start, lr_end = float(train.iloc[0]["lr"]), float(train.iloc[-1]["lr"])
    def finite_or_none(value):
        return float(value) if pd.notna(value) and np.isfinite(float(value)) else None
    def metric_text(value):
        return f"{value:.4f}" if value is not None else "undefined (zero denominator)"
    class_iou = {c: finite_or_none(latest[f"iou_{c}"]) for c in CLASSES}
    class_prec = {c: finite_or_none(latest[f"precision_{c}"]) for c in CLASSES}
    class_rec = {c: finite_or_none(latest[f"recall_{c}"]) for c in CLASSES}
    plateau = recent_range < 0.005 if len(recent) == 5 else None
    lines = [
        "# SegFormer baseline — training behaviour report" if full else "# SegFormer baseline — INTERIM training behaviour report",
        "",
        f"**Scope:** fixed seed 42, validation only; epochs {first_epoch}–{last_epoch} ({len(val)}/20). The test sets were not accessed by this analysis.",
        "",
        "## Convergence, underfitting and overfitting",
        f"- Validation mIoU4 changed from {val.iloc[0]['miou4']:.4f} to {latest['miou4']:.4f} (Δ {d_miou:+.4f}). Best logged mIoU4 was {bestrow['miou4']:.4f} in epoch {int(bestrow['epoch'])}.",
        f"- Training batch-mean loss changed by {d_train:+.4f} between the first and last logged epoch means. Validation labelled-pixel-mean loss changed by {d_val:+.4f}.",
        "- Training loss averages batches while validation loss is weighted by labelled pixels; their absolute difference is only an approximate generalisation-gap indicator, not an identical loss estimator.",
    ]
    if gap_points:
        lines += [f"- Approximate validation-minus-training loss gap: {gap_points[0][1]:+.4f} in epoch {gap_points[0][0]}, {gap_points[-1][1]:+.4f} in epoch {gap_points[-1][0]}."]
    if plateau is not None:
        lines += [f"- Last-five-epoch validation mIoU4 range: {recent_range:.4f}; net change: {recent_change:+.4f}. " +
                  ("This is consistent with a possible plateau (illustrative <0.005 range criterion), not proof of convergence." if plateau
                   else "The latest five epochs have not met the illustrative <0.005 range plateau criterion.")]
    else:
        lines += ["- Fewer than five validation epochs are available; a late-training plateau cannot yet be assessed."]
    lines += [
        "- Overfitting would be suggested by persistently improving training loss alongside worsening validation loss/mIoU; underfitting by weak performance on both. Interpret the curves jointly before assigning a firm diagnosis.",
        "",
        "## Per-class performance at latest validation epoch",
    ]
    for label, c in zip(DISPLAY, CLASSES):
        lines += [f"- {label}: IoU {metric_text(class_iou[c])}; precision {metric_text(class_prec[c])}; recall {metric_text(class_rec[c])}."]
    lines += [
        "- **Big Rock** is rare in the supplied training-label audit (~0.85% of labelled pixels). Its IoU, recall, and confusion-matrix row deserve separate examination; aggregate mIoU can mask minority-class instability.",
        "",
        "## Bias and variance limitations",
        "- The recorded train/validation curves describe behaviour of **one** fixed-seed run and one in-domain validation split. They do not estimate seed-to-seed variance, confidence intervals, or cross-rover generalisation.",
        "- Any bias/variance interpretation is qualitative. Estimating stochastic variance would require additional seeds, outside this locked experiment.",
        "",
        "## Learning rate, AMP and stability",
        f"- Logged learning rate: {lr_start:.7g} at iteration {int(train.iloc[0]['iteration'])} to {lr_end:.7g} at iteration {int(train.iloc[-1]['iteration'])}; examine the plotted 100-step warmup and remaining cosine schedule.",
        f"- Recorded GradScaler overflow/skipped steps: {skipped} total (at first logged step: {skip_first})." + (f" Recorded nonfinite gradient norms: {nfg}." if nfg is not None else ""),
        "- A nonfinite gradient norm alone is **not** used as proof of an AMP-skipped optimiser step; the runner separately monitors GradScaler scale decreases.",
        "- Check abrupt loss spikes, skipped-step clusters, unexpected LR discontinuities, and class-specific oscillations before interpreting stability.",
        "",
        "## Decision boundary",
        "- This is an observational report, not a new hyperparameter-selection rule. No test-set result is involved and no change to the frozen protocol is recommended from these diagnostics alone.",
    ]
    summary = dict(complete=bool(full), epochs=int(len(val)), best_epoch=int(bestrow["epoch"]),
                   best_val_miou4=float(bestrow["miou4"]), last_val_miou4=float(latest["miou4"]),
                   last_val_loss=float(latest["val_loss"]), last_iou=class_iou,
                   latest_precision=class_prec, latest_recall=class_rec,
                   skipped_amp_steps_total=skipped,
                   nonfinite_grad_norm_total=nfg,
                   last_5_epoch_miou4_range=recent_range if len(recent) == 5 else None,
                   observed_plateau_by_illustrative_threshold=plateau)
    return "\n".join(lines) + "\n", summary


def analyze(root):
    root = Path(root)
    base = root / "results" / "baseline"
    out = base / "training_analysis"
    train_path = base / "train_log.csv"
    val_path = base / "validation_log.csv"
    if not train_path.is_file() or not val_path.is_file():
        raise FileNotFoundError("Local training/validation CSVs not found; no analysis is possible before training")
    train = pd.read_csv(train_path).sort_values("iteration")
    val = pd.read_csv(val_path).sort_values("iteration")
    if train.empty or val.empty:
        raise ValueError("Local CSVs are empty")
    for frame, tag, fields in [(train, "train", ["iteration", "epoch", "loss", "lr"]),
                               (val, "validation", ["iteration", "epoch", "val_loss", "miou3", "miou4", "pixacc"] +
                                [f"{metric}_{c}" for metric in ("iou", "precision", "recall") for c in CLASSES])]:
        missing = sorted(set(fields) - set(frame.columns))
        if missing:
            raise ValueError(f"Missing {tag} columns: {missing}")
        if frame["iteration"].duplicated().any() or not frame["iteration"].is_monotonic_increasing:
            raise ValueError(f"Duplicate/out-of-order {tag} iterations; reconcile before plotting")
    out.mkdir(parents=True, exist_ok=True)
    _line(out, train["epoch"], [("Training loss", train["loss"])],
          "Training loss by iteration (displayed against epoch)", "Training loss", "training_loss_vs_epoch.png")
    _line(out, val["epoch"], [("Validation loss", val["val_loss"])],
          "Validation loss by epoch", "Validation loss", "validation_loss_vs_epoch.png")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train["epoch"], train["loss"], label="Train (batch mean)", alpha=0.7)
    ax.plot(val["epoch"], val["val_loss"], label="Validation (labelled-pixel mean)", marker="o")
    ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Train and validation loss (different averaging conventions)")
    ax.grid(alpha=0.25)
    ax.legend()
    _save(out / "combined_training_validation_loss.png")
    _line(out, val["epoch"], [("mIoU3", val["miou3"]), ("mIoU4", val["miou4"])],
          "Validation mean IoU", "mIoU", "miou3_miou4_vs_epoch.png")
    for metric, title in [("iou", "IoU"), ("precision", "Precision"), ("recall", "Recall")]:
        _line(out, val["epoch"], [(label, val[f"{metric}_{c}"]) for label, c in zip(DISPLAY, CLASSES)],
              f"Per-class validation {title}", title,
              f"per_class_{metric}_vs_epoch.png")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train["iteration"], train["lr"])
    ax.set(xlabel="Global iteration", ylabel="Learning rate", title="Learning rate vs global iteration")
    ax.grid(alpha=0.25)
    _save(out / "learning_rate_vs_iteration.png")
    cm_dir = base / "confusion_matrices"
    for _, r in val.iterrows():
        epoch = int(r["epoch"])
        cm_path = cm_dir / f"epoch_{epoch:02d}.json"
        if not cm_path.is_file():
            raise FileNotFoundError(f"Missing validation confusion matrix: {cm_path}")
        payload = json.loads(cm_path.read_text())
        if int(payload["iteration"]) != int(r["iteration"]):
            raise ValueError(f"Confusion matrix epoch {epoch} mismatches validation CSV")
        _confusion_plot(out / f"confusion_epoch_{epoch:02d}.png", payload["confusion_matrix"], epoch)
    report, summary = _build_report(train, val)
    (out / "training_behaviour_report.md").write_text(report, encoding="utf8")
    (out / "analysis_summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf8")
    print(f"Analysis complete ({len(val)} validation epochs; W&B not used). Output: {out}")
    return summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", default="/kaggle/working/mars_work")
    args = parser.parse_args()
    analyze(args.root)


Writing /kaggle/working/mars_work/code/baseline_analysis.py
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [28]:
%%writefile /kaggle/working/mars_work/code/verify_frozen_split.py
"""Read-only reproduction of Backup 10's original frozen-split canonical MD5 check.

The frozen descriptor MD5 is NOT the hash of the current raw CSV bytes.
Original notebook procedure: pandas read_csv -> remove img_512/mask_512 ->
to_csv(index=False).encode() -> MD5. Never modifies JSON/CSV/dataset.
"""
import argparse
import hashlib
import json
from pathlib import Path

import pandas as pd

SPLITS = ("train", "val", "test_msl", "test_mer")


def verify(dataset_root, split_json):
    root = Path(dataset_root)
    sp = json.loads(Path(split_json).read_text(encoding="utf8"))
    if sp.get("status") != "FROZEN":
        raise ValueError("Frozen descriptor status is not FROZEN")
    results = {}
    for name in SPLITS:
        csv_path = root / f"{name}.csv"
        df = pd.read_csv(csv_path)
        missing = set(("img_512", "mask_512")) - set(df.columns)
        if missing:
            raise ValueError(f"{name}.csv missing expected appended columns: {missing}")
        original = df.drop(columns=["img_512", "mask_512"])
        actual = hashlib.md5(original.to_csv(index=False).encode()).hexdigest()
        expected = sp["files"][name]["md5"]
        expected_rows = sp["files"][name]["rows"]
        raw_md5 = hashlib.md5(csv_path.read_bytes()).hexdigest()
        ok = actual == expected and len(df) == expected_rows
        results[name] = ok
        print(f"{name:9} rows {len(df):5}/{expected_rows:5} | raw MD5 matches={raw_md5 == expected} "
              f"| canonical MD5={actual} | expected={expected} | {'PASS' if ok else 'FAIL'}")
    print("FROZEN SPLIT CANONICAL HASHES:", "ALL FOUR PASS" if all(results.values()) else "MISMATCH; STOP")
    return results


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="/tmp/mars_data")
    ap.add_argument("--split", default="/kaggle/working/mars_work/splits/split_v3_FROZEN.json")
    a = ap.parse_args()
    if not all(verify(a.data, a.split).values()):
        raise SystemExit(1)


Writing /kaggle/working/mars_work/code/verify_frozen_split.py


In [29]:
%%writefile /kaggle/working/mars_work/code/baseline_checks.py
"""Read-only non-training smoke/integration checks of new baseline code.

Synthetic test data are created under tempfile.TemporaryDirectory, NEVER in
/kaggle/working/mars_work, and no GPU training or backups run here.
"""
import csv
import importlib
import importlib.util
import json
import math
import os
import sys
import tempfile
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch

CODE = Path(__file__).resolve().parent
sys.path.insert(0, str(CODE))
import baseline_train as bt
import baseline_analysis as ba
import verify_frozen_split as vs
import ckpt_utils as cu
import tracking as tk


def check(name, fn):
    fn()
    print("PASS:", name)


def frozen_hash_test(tmp):
    root = tmp / "d"; root.mkdir()
    desc = {"status": "FROZEN", "files": {}}
    for name in vs.SPLITS:
        df = pd.DataFrame({"image_path": ["a.png", "b.png"], "label_path": ["a.png", "b.png"],
                           "class_0": [0.5, 0.8], "img_512": ["img/a.png", "img/b.png"],
                           "mask_512": ["mask/a.png", "mask/b.png"]})
        csv_path = root / f"{name}.csv"
        df.to_csv(csv_path, index=False)
        orig = df.drop(columns=["img_512", "mask_512"])
        import hashlib
        desc["files"][name] = {"rows": len(df), "md5": hashlib.md5(orig.to_csv(index=False).encode()).hexdigest()}
        assert hashlib.md5(csv_path.read_bytes()).hexdigest() != desc["files"][name]["md5"]
    p = tmp / "split.json"; p.write_text(json.dumps(desc))
    assert all(vs.verify(root, p).values())


def tracker_test(tmp):
    class FakeRun:
        def __init__(self): self.logs = []; self.finished = False
        def log(self, data, step=None): self.logs.append((step, data))
        def finish(self): self.finished = True
    t = tk.Tracker(tmp / "train", use_wandb=False)
    fake = FakeRun(); t.run = fake
    v = tk.Tracker(tmp / "validation", use_wandb=False)
    row = dict(iteration=25, epoch=25/1225, loss=.6, lr=.0001,
               grad_norm=.8, skipped_steps_total=0, nonfinite_grad_norm_total=0, elapsed_s=5.)
    t.log("log", row, step=25)
    vrow = dict(iteration=1225, epoch=1, val_loss=.4, miou3=.5, miou4=.4,
                pixacc=.9, eval_s=9., epoch_duration_s=100.,
                epoch_mean_train_loss=.5, is_best=True)
    for c in bt.CN:
        for key in ["iou", "precision", "recall"]: vrow[f"{key}_{c}"] = .5
    v.log("log", vrow, step=1225)
    epoch_row = dict(row, iteration=1225, epoch=1)
    t.log("log", {**epoch_row, **bt.wb_validation_scalars(vrow)}, step=1225)
    assert [x[0] for x in fake.logs] == [25, 1225]
    wb = fake.logs[-1][1]
    assert all(f"log/validation_{x}" in wb for x in ("miou3", "miou4", "val_loss", "pixacc", "epoch_duration_s", "iou_bigrock", "precision_soil", "recall_sand"))
    assert wb["log/best_checkpoint_event"] == 1
    raw = pd.read_csv(tmp / "train_log.csv")
    assert list(raw.columns) == list(row.keys()) and len(raw) == 2
    assert "validation_miou4" not in raw.columns
    assert "miou4" in pd.read_csv(tmp / "validation_log.csv")
    t.finish(); v.finish(); assert fake.finished

    # Same fixed CSV schema on restart, including if first resumed row lands on epoch end.
    restarted = tk.Tracker(tmp / "train", use_wandb=False)
    bt.resume_tracker_csv_fields(restarted)
    restarted.log("log", {**dict(row, iteration=2450, epoch=2), **bt.wb_validation_scalars(vrow)}, step=2450)
    loaded = pd.read_csv(tmp / "train_log.csv")
    assert list(loaded.columns) == list(row.keys()) and len(loaded) == 3
    class FailingRun:
        def log(self, *a, **kw): raise OSError("mock disconnection")
        def finish(self): pass
    restarted.run = FailingRun()
    restarted.log("log", dict(row, iteration=2475, epoch=2475/1225), step=2475)
    assert restarted.wb_errors == 1 and len(pd.read_csv(tmp / "train_log.csv")) == 4


def resume_test(tmp):
    cfg = bt.build_config()
    assert cfg["epochs"] == 20 and cfg["total_iters"] == 24500 and cfg["start_lr"] == .0001
    assert bt.STAGE1_END == 10 * bt.SPE and bt.TOTAL == 20 * bt.SPE
    model = torch.nn.Linear(2, 1)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["start_lr"], weight_decay=cfg["weight_decay"])
    sched = cu.make_scheduler(opt, bt.TOTAL, 100, .05)
    scaler = torch.amp.GradScaler("cpu", enabled=False)
    for _ in range(50):
        opt.zero_grad()
        model(torch.ones(1, 2)).sum().backward(); opt.step(); sched.step()
    path = tmp / "last.pt"
    cu.save_checkpoint(path, model, opt, sched, scaler, 0, 50, cfg, .55,
                       extra={"skipped": 0, "epoch_seen": 50, "epoch_loss_sum": 23., "epoch_elapsed_s": 100.})
    lr_at_ck = float(opt.param_groups[0]["lr"])
    fresh = torch.nn.Linear(2, 1)
    opt2 = torch.optim.AdamW(fresh.parameters(), lr=cfg["start_lr"], weight_decay=cfg["weight_decay"])
    sched2 = cu.make_scheduler(opt2, bt.TOTAL, 100, .05)
    ck = cu.load_checkpoint(path, fresh, opt2, sched2, scaler, cfg=cfg, restore_rng=True)
    assert ck["iteration"] == 50 and ck["extra"]["epoch_seen"] == 50
    assert sched2.last_epoch == sched.last_epoch and math.isclose(opt2.param_groups[0]["lr"], lr_at_ck)
    for a, b in zip(model.parameters(), fresh.parameters()): assert torch.equal(a, b)
    opt.zero_grad(); model(torch.ones(1, 2)).sum().backward(); opt.step(); sched.step()
    opt2.zero_grad(); fresh(torch.ones(1, 2)).sum().backward(); opt2.step(); sched2.step()
    assert sched.last_epoch == sched2.last_epoch
    assert math.isclose(opt.param_groups[0]["lr"], opt2.param_groups[0]["lr"])
    # Simulate logs emitted beyond a last.pt checkpoint; reconcile and write again.
    csv_path = tmp / "train_log.csv"
    with csv_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["iteration", "epoch", "loss"]); w.writeheader()
        for it in (25, 50, 75, 100): w.writerow(dict(iteration=it, epoch=it/1225, loss=.5))
    assert bt.truncate_csv(csv_path, 50) == 2
    t = tk.Tracker(tmp / "train", use_wandb=False)
    bt.resume_tracker_csv_fields(t)
    t.log("log", dict(iteration=75, epoch=75/1225, loss=.4), step=75)
    rows = list(csv.DictReader(csv_path.open()))
    assert [int(x["iteration"]) for x in rows] == [25, 50, 75]
    # Best checkpoint selection/pointer immutability & orphan recovery.
    ck_dir = tmp / "checkpoints"; ck_dir.mkdir()
    old = ck_dir / "best_epoch01_it1225.pt"; old.write_bytes(b"fake")
    moved = bt.recover_best_files(ck_dir, 1000, {}, -1., float("inf"))
    assert moved == [old.name] and not old.exists() and (ck_dir/"orphans"/old.name).exists()
    committed = ck_dir / "best_epoch01_it1000.pt"; committed.write_bytes(b"fake committed")
    bt.update_best_pointer(ck_dir, committed.name, 1000, .72, .14, 1)
    assert (ck_dir / "best.pt").read_bytes() == committed.read_bytes()
    assert bt.compare_best(.8, .2, 2, .7, .1, 1)
    assert not bt.compare_best(.7, .3, 2, .7, .2, 1)
    assert cu.lr_factor(12251, bt.TOTAL) < cu.lr_factor(12250, bt.TOTAL)
    assert cu.lr_factor(bt.TOTAL, bt.TOTAL) == .05
    # Undefined class precision is valid and must not corrupt JSON.
    assert bt.json_numbers([.9, float('nan'), 0.0]) == [.9, None, 0.0]
    json.dumps(bt.json_numbers([.9, float('nan'), 0.0]), allow_nan=False)



def two_stage_synthetic_test(tmp):
    """Actually execute run() with a tiny CPU model, not Mars GPU or test images."""
    old = dict(TOTAL=bt.TOTAL, STAGE1_END=bt.STAGE1_END, SPE=bt.SPE,
               build_config=bt.build_config, preflight=bt.preflight,
               new_model=bt.mc.new_model, MarsSeg=bt.mc.MarsSeg,
               train_step=bt.mc.train_step, evaluate=bt.mc.evaluate,
               make_det_loader=bt.md.make_det_loader)
    old_env = os.environ.get("MARS_USE_WANDB")
    os.environ["MARS_USE_WANDB"] = "0"
    root = tmp / "project"
    (root / "checkpoints/lr_pilot").mkdir(parents=True)
    calls = {"eval": 0}
    try:
        bt.TOTAL = 4; bt.STAGE1_END = 2; bt.SPE = 2
        def mini_config():
            cfg = old["build_config"]()
            cfg.update(epochs=2, total_iters=4, steps_per_epoch=2,
                       train_log_every=1, checkpoint_every=1, workers=0,
                       val_batch_size=2)
            return cfg
        bt.build_config = mini_config
        bt.preflight = lambda root, cfg: (16, 4)
        bt.mc.new_model = lambda: torch.nn.Linear(2, 1)
        torch.manual_seed(42)
        first = torch.nn.Linear(2,1)
        torch.save(dict(state_hash=mini_config()["init_state_hash"],
                        state_dict=first.state_dict()),
                   root / "checkpoints/lr_pilot/init_seed42.pt")
        bt.mc.MarsSeg = lambda split: [(torch.ones(1,2), torch.ones(1,1)) for _ in range(4)]
        def train_step(model, opt, scaler, x, y):
            opt.zero_grad(set_to_none=True)
            pred = model(x)
            loss = ((pred - y)**2).mean()
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9)
            opt.step()
            return float(loss), float(gn)
        bt.mc.train_step = train_step
        def evaluate(model, loader):
            calls["eval"] += 1
            m = .55 + .08 * calls["eval"]
            return dict(miou4=m, miou3=m+.05, loss=.4-.05*calls["eval"],
                        pixacc=.8, iou=[.7,.65,.6,.2+calls["eval"]*.1],
                        precision=[.8,.8,.8,float('nan') if calls["eval"]==1 else .4],
                        recall=[.8,.8,.8,.4],
                        conf=[[20,1,0,0],[0,20,1,0],[0,1,20,0],[0,2,3,5]])
        bt.mc.evaluate = evaluate
        bt.md.make_det_loader = lambda seed, bs, start_iter, end_iter, **kw: [
            (torch.ones(1,2), torch.ones(1,1)) for _ in range(start_iter,end_iter)]
        bt.run(root, 1)
        ck = torch.load(root / "checkpoints/baseline/last.pt", map_location="cpu", weights_only=False)
        assert ck["iteration"] == 2 and ck["epoch"] == 1
        assert len(pd.read_csv(root / "results/baseline/validation_log.csv")) == 1
        bt.run(root, 2)
        ck2 = torch.load(root / "checkpoints/baseline/last.pt", map_location="cpu", weights_only=False)
        assert ck2["iteration"] == 4 and ck2["epoch"] == 2
        tr = pd.read_csv(root / "results/baseline/train_log.csv")
        va = pd.read_csv(root / "results/baseline/validation_log.csv")
        assert tr["iteration"].tolist() == [1,2,3,4]
        assert va["iteration"].tolist() == [2,4]
        assert va["is_best"].tolist() == [True, True]
        assert va["epoch_duration_s"].gt(0).all()
        assert (root / "checkpoints/baseline/best_epoch02_it4.pt").is_file()
        cm1 = json.loads((root / "results/baseline/confusion_matrices/epoch_01.json").read_text())
        assert cm1["precision"][-1] is None  # undefined precision cannot break JSON
        summary = ba.analyze(root)
        assert summary["epochs"] == 2
    finally:
        bt.TOTAL=old["TOTAL"]; bt.STAGE1_END=old["STAGE1_END"]; bt.SPE=old["SPE"]
        bt.build_config=old["build_config"]; bt.preflight=old["preflight"]
        bt.mc.new_model=old["new_model"]; bt.mc.MarsSeg=old["MarsSeg"]
        bt.mc.train_step=old["train_step"]; bt.mc.evaluate=old["evaluate"]
        bt.md.make_det_loader=old["make_det_loader"]
        if old_env is None: os.environ.pop("MARS_USE_WANDB", None)
        else: os.environ["MARS_USE_WANDB"] = old_env


def wandb_resume_test(tmp):
    import importlib.machinery
    fake = types.ModuleType("wandb")
    fake.__spec__ = importlib.machinery.ModuleSpec("wandb", loader=None)
    class Run:
        def __init__(self): self.logs=[]
        def log(self,data,step=None): self.logs.append((step,data))
        def finish(self): pass
    fake.init = lambda **kwargs: Run()
    fake.login = lambda **kwargs: None
    old = sys.modules.get("wandb")
    sys.modules["wandb"] = fake
    env_keys = ["WANDB_MODE", "WANDB_RUN_ID", "WANDB_RESUME", "WANDB_API_KEY", "WANDB_DIR", "MARS_USE_WANDB"]
    old_env = {k: os.environ.get(k) for k in env_keys}
    try:
        os.environ["WANDB_MODE"] = "offline"
        os.environ["MARS_USE_WANDB"] = "1"
        os.environ.pop("WANDB_API_KEY", None)
        res = tmp / "res"; res.mkdir()
        enabled, state = bt.prepare_wandb(res, bt.build_config(), 0)
        assert enabled and len(state["run_id"]) == 16
        t = tk.Tracker(res / "train", project="test", use_wandb=enabled)
        assert t.run is not None
        t.log("log", dict(iteration=25, loss=.5), step=25)
        assert t.run.logs[0][0] == 25
        t.finish()
        state["last_submitted_iteration"] = 100
        bt.atomic_json(res / "wandb_run.json", state)
        same_enabled, same = bt.prepare_wandb(res, bt.build_config(), 100)
        assert same_enabled and same["run_id"] == state["run_id"]
        _, forked = bt.prepare_wandb(res, bt.build_config(), 50)
        assert forked["run_id"] != state["run_id"]
        assert forked["previous_run_id"] == state["run_id"]
    finally:
        for k,v in old_env.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k]=v
        if old is None: sys.modules.pop("wandb", None)
        else: sys.modules["wandb"] = old


def analysis_test(tmp):
    base = tmp / "results/baseline"; base.mkdir(parents=True)
    train = []
    for it in range(25, 3675+1, 25):
        ep = it / 1225
        train.append(dict(iteration=it, epoch=ep, loss=max(.1, .8-.15*ep), lr=.0001*max(.05,1-ep/20),
                          grad_norm=1., skipped_steps_total=0, nonfinite_grad_norm_total=0,
                          elapsed_s=it*0.05))
    pd.DataFrame(train).to_csv(base / "train_log.csv", index=False)
    val=[]; cm = base / "confusion_matrices"; cm.mkdir()
    for epoch in range(1,4):
        it = epoch*1225
        row = dict(iteration=it, epoch=epoch, val_loss=.7-.1*epoch,
                   miou3=.45+.03*epoch, miou4=.40+.035*epoch, pixacc=.7+.04*epoch,
                   eval_s=3., epoch_duration_s=60., epoch_mean_train_loss=.8-.15*epoch,
                   skipped_steps_total=0, is_best=True)
        for c in bt.CN:
            for k in ("iou", "precision", "recall"):
                row[f"{k}_{c}"] = .2 + .1*epoch
        val.append(row)
        cmatrix = [[100,2,3,0],[2,100,3,0],[0,3,80,2],[1,2,3,15]]
        (cm / f"epoch_{epoch:02d}.json").write_text(json.dumps(dict(iteration=it,epoch=epoch,confusion_matrix=cmatrix)))
    pd.DataFrame(val).to_csv(base / "validation_log.csv", index=False)
    summary = ba.analyze(tmp)
    out = base / "training_analysis"
    files = list(out.glob("*.png"))
    assert len(files) == 8+3, [x.name for x in files]
    assert (out / "training_behaviour_report.md").is_file()
    assert (out / "analysis_summary.json").is_file()
    assert summary["epochs"] == 3 and not summary["complete"]
    assert "one" in (out / "training_behaviour_report.md").read_text()


if __name__ == "__main__":
    with tempfile.TemporaryDirectory(prefix="mars_baseline_check_") as d:
        tmp = Path(d)
        for sub in ("hashes", "tracker", "resume", "wandb", "analysis", "two_stage"):
            (tmp / sub).mkdir()
        check("Canonical frozen split hashing (synthetic, all four)", lambda: frozen_hash_test(tmp / "hashes"))
        check("Original Tracker: shared W&B step, local CSV, failed W&B, resume CSV schema", lambda: tracker_test(tmp / "tracker"))
        check("Checkpoint / scheduler / log rollback / best pointer on synthetic CPU model", lambda: resume_test(tmp / "resume"))
        check("Synthetic two-stage end-to-end runner (no Mars GPU training)", lambda: two_stage_synthetic_test(tmp / "two_stage"))
        check("Optional W&B mock login, run ID continuity and rewind fork", lambda: wandb_resume_test(tmp / "wandb"))
        check("All 8 figure types, per-epoch confusion plots and interim report", lambda: analysis_test(tmp / "analysis"))
    print("ALL NON-TRAINING CHECKS PASSED. No Kaggle GPU training, test split evaluation or backup was run.")


Writing /kaggle/working/mars_work/code/baseline_checks.py


In [30]:
import subprocess, sys
subprocess.run([sys.executable, "/kaggle/working/mars_work/code/verify_frozen_split.py",
                "--data", "/tmp/mars_data",
                "--split", "/kaggle/working/mars_work/splits/split_v3_FROZEN.json"], check=True)

train     rows  9803/ 9803 | raw MD5 matches=False | canonical MD5=8d02c9c95be90e4a89663a84c3fd82b7 | expected=8d02c9c95be90e4a89663a84c3fd82b7 | PASS
val       rows  1192/ 1192 | raw MD5 matches=False | canonical MD5=0a86468c938bf3009b6db117c881c38d | expected=0a86468c938bf3009b6db117c881c38d | PASS
test_msl  rows   311/  311 | raw MD5 matches=False | canonical MD5=46a63b56c865a1d2c6c8bc13a90bdeaa | expected=46a63b56c865a1d2c6c8bc13a90bdeaa | PASS
test_mer  rows   195/  195 | raw MD5 matches=False | canonical MD5=289521ca2a432d9cb63c6fb8138a5f41 | expected=289521ca2a432d9cb63c6fb8138a5f41 | PASS
FROZEN SPLIT CANONICAL HASHES: ALL FOUR PASS


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mars_work/code/verify_frozen_split.py', '--data', '/tmp/mars_data', '--split', '/kaggle/working/mars_work/splits/split_v3_FROZEN.json'], returncode=0)

In [31]:
import subprocess, sys
subprocess.run([sys.executable, "/kaggle/working/mars_work/code/baseline_checks.py"], check=True)

train     rows     2/    2 | raw MD5 matches=False | canonical MD5=ad76628f941a86ea62501cfabf651ecc | expected=ad76628f941a86ea62501cfabf651ecc | PASS
val       rows     2/    2 | raw MD5 matches=False | canonical MD5=ad76628f941a86ea62501cfabf651ecc | expected=ad76628f941a86ea62501cfabf651ecc | PASS
test_msl  rows     2/    2 | raw MD5 matches=False | canonical MD5=ad76628f941a86ea62501cfabf651ecc | expected=ad76628f941a86ea62501cfabf651ecc | PASS
test_mer  rows     2/    2 | raw MD5 matches=False | canonical MD5=ad76628f941a86ea62501cfabf651ecc | expected=ad76628f941a86ea62501cfabf651ecc | PASS
FROZEN SPLIT CANONICAL HASHES: ALL FOUR PASS
PASS: Canonical frozen split hashing (synthetic, all four)
  W&B log failed (training continues): OSError('mock disconnection')
PASS: Original Tracker: shared W&B step, local CSV, failed W&B, resume CSV schema
PASS: Checkpoint / scheduler / log rollback / best pointer on synthetic CPU model
Starting approved 20-epoch baseline from shared init seed 4

baseline stage 1:   0%|          | 0/2 [00:00<?, ?it/s]/kaggle/working/mars_work/code/baseline_checks.py:186: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), float(gn)
baseline stage 2: 100%|██████████| 2/2 [00:00<00:00, 81.60it/s, iter=4, loss=0.001, lr=4e-06]


STAGE 2 COMPLETE | 4/4 iterations | best val mIoU4=0.710000
STOP: freeze best checkpoint before the one-time final test evaluation.
Analysis complete (2 validation epochs; W&B not used). Output: /tmp/mars_baseline_check_4ltvymb6/two_stage/project/results/baseline/training_analysis
PASS: Synthetic two-stage end-to-end runner (no Mars GPU training)
W&B history exceeded committed checkpoint; new grouped W&B segment for safe rewind
PASS: Optional W&B mock login, run ID continuity and rewind fork
Analysis complete (3 validation epochs; W&B not used). Output: /tmp/mars_baseline_check_4ltvymb6/analysis/results/baseline/training_analysis
PASS: All 8 figure types, per-epoch confusion plots and interim report
ALL NON-TRAINING CHECKS PASSED. No Kaggle GPU training, test split evaluation or backup was run.


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mars_work/code/baseline_checks.py'], returncode=0)

In [32]:
import os, sys, importlib.util
from pathlib import Path
sys.path.insert(0, "/kaggle/working/mars_work/code")
import tracking as tracking_original

have_key = bool(os.getenv("WANDB_API_KEY"))
if not have_key:
    try:
        from kaggle_secrets import UserSecretsClient
        have_key = bool(UserSecretsClient().get_secret("WANDB_API_KEY"))
    except Exception:
        pass
mode = os.getenv("WANDB_MODE", "online").lower()
if importlib.util.find_spec("wandb") is None or (not have_key and mode not in ("offline", "dryrun")):
    print("W&B smoke test SKIPPED: package or secure key unavailable; local CSV remains authoritative")
else:
    smoke = tracking_original.Tracker(Path("/tmp/mars_wandb_preflight"),
        project="mars-terrain-segmentation", name="preflight-only-not-training",
        group="baseline-preflight", use_wandb=True,
        config={"purpose": "connection_smoke_test_only"})
    smoke.log("smoke", {"iteration": 0, "preflight_only": 1}, step=0)
    print("W&B smoke:", "SDK run initialized" if smoke.run is not None else "offline/local-only")
    print("W&B log errors:", smoke.wb_errors)
    smoke.finish()
    print("Local smoke log:", "/tmp/mars_wandb_preflight_smoke.csv")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hudafatimah-04 (hudafatimah-04-rv-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B smoke: SDK run initialized
W&B log errors: 0


smoke/iteration,▁
smoke/preflight_only,▁
smoke/iteration,0
smoke/preflight_only,1


Local smoke log: /tmp/mars_wandb_preflight_smoke.csv


In [33]:
import subprocess, sys
subprocess.run([sys.executable, "/kaggle/working/mars_work/code/baseline_train.py", "--check"], check=True)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 11584.64it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias

PREFLIGHT PASSED: train=9803, val=1192, config_hash=2e177076344b5638, CUDA=Tesla T4
No training was performed; no Backup 11/12 was created.


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mars_work/code/baseline_train.py', '--check'], returncode=0)

In [34]:

from pathlib import Path
import sys, json, hashlib, zipfile, subprocess

W = Path("/kaggle/working/mars_work")
sys.path.insert(0, str(W / "code"))
import baseline_train as bt

# Preserve the approved baseline configuration.
cfg_file = W / "configs/baseline_protocol.json"
cfg = bt.build_config()

if cfg_file.exists():
    assert json.loads(cfg_file.read_text()) == cfg
else:
    bt.atomic_json(cfg_file, cfg)

def save_baseline_backup(name):
    output = Path("/kaggle/working") / f"{name}.zip"
    if output.exists():
        raise FileExistsError(f"Backup already exists: {output}")

    files = []
    for folder in ("code", "configs", "splits",
                   "checkpoints", "results", "logs"):
        base = W / folder
        if base.exists():
            files.extend(
                p for p in base.rglob("*")
                if p.is_file()
                and not any(x in {"__pycache__", "wandb_runs",
                                  "wandb", "orphans"}
                            for x in p.relative_to(W).parts)
                and not p.name.endswith((".tmp", ".part"))
            )

    for name0 in ("MANIFEST.json", "MANIFEST_backup09.json"):
        p = W / name0
        if p.is_file():
            files.append(p)

    manifest = {}
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED,
                         compresslevel=1, allowZip64=True) as z:
        for p in sorted(set(files)):
            rel = p.relative_to(W).as_posix()
            with p.open("rb") as f:
                manifest[rel] = hashlib.file_digest(f, "sha256").hexdigest()
            z.write(p, rel)

        z.writestr("BACKUP_MANIFEST.json",
                   json.dumps({"files": manifest}, indent=2))

    with zipfile.ZipFile(output) as z:
        assert z.testzip() is None, "ZIP integrity failed"
        assert set(manifest).issubset(set(z.namelist()))

    print(f"VERIFIED: {output}")
    print(f"Files: {len(manifest)} | Size: {output.stat().st_size:,} bytes")
    return output

backup11 = save_baseline_backup("backup_11_baseline_ready")

print("\nSTARTING ACTUAL TRAINING: EPOCHS 1–10")
subprocess.run(
    [sys.executable, str(W / "code/baseline_train.py"), "--stage", "1"],
    check=True
)


VERIFIED: /kaggle/working/backup_11_baseline_ready.zip
Files: 50 | Size: 139,422,690 bytes

STARTING ACTUAL TRAINING: EPOCHS 1–10


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 15302.44it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.runn

Starting approved 20-epoch baseline from shared init seed 42
W&B: connected


baseline stage 1:  10%|▉         | 1224/12250 [03:02<27:16,  6.74it/s, iter=1224, loss=0.066, lr=0.0001]

epoch 01/20 | val mIoU4=0.714713 | mIoU3=0.894504 | val loss=0.130834 | best epoch=1


baseline stage 1:  20%|█▉        | 2449/12250 [06:14<24:20,  6.71it/s, iter=2449, loss=0.042, lr=9.8e-05]  

... (16 outputs left out)

epoch 10/20 | val mIoU4=0.818236 | mIoU3=0.924224 | val loss=0.093140 | best epoch=7


baseline stage 1: 100%|██████████| 12250/12250 [31:51<00:00,  6.41it/s, iter=12250, loss=0.069, lr=5.3e-05]
wandb: updating run metadata
wandb: uploading config.yaml; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:       log/best_checkpoint_event ██▁█▁▁█▁▁▁
wandb:                   log/elapsed_s ▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇██
wandb:                       log/epoch ▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
wandb:                   log/grad_norm █▆█▆▅▄▄▅▃▃▃▂▂▄▃▄▅▃▄▃▂▂▄▂▂▂▃▂▂▂▁▂▁▃▂▂▂▁▂▂
wandb:                   log/iteration ▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█
wandb:                        log/loss █▄▃▃▃▃▃▃▂▂▂▂▂▂▃▂▂▂▁▁▂▂▁▂▁▁▂▁▁▁▁▁▂▁▁▁▁▁▁▁
wandb:                          log/lr ██████████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▁▁
wandb:   log/nonfinite_grad_norm_total ▁▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████
wandb:         log/skipped_steps_total ▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆██████
wandb: log/validation_epoch_duration_s █▆▅▄▂▂▃▁▂▁
wandb:                             +

STAGE 1 COMPLETE | 12250/24500 iterations | best val mIoU4=0.829310
STOP: create/verify genuine Backup 12 before running --stage 2.


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mars_work/code/baseline_train.py', '--stage', '1'], returncode=0)

In [35]:

import torch

W = Path("/kaggle/working/mars_work")
last = W / "checkpoints/baseline/last.pt"

ck = torch.load(last, map_location="cpu", weights_only=False)

assert ck["iteration"] == 12250
assert ck["config_hash"] == bt.cu.config_hash(bt.build_config())

print("Epoch 10 checkpoint VERIFIED")

backup12 = save_baseline_backup("backup_12_epoch10")
print("BACKUP 12 READY:", backup12)


Epoch 10 checkpoint VERIFIED
VERIFIED: /kaggle/working/backup_12_epoch10.zip
Files: 70 | Size: 387,995,508 bytes
BACKUP 12 READY: /kaggle/working/backup_12_epoch10.zip


In [36]:

import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "/kaggle/working/mars_work/code/baseline_train.py",
        "--stage", "2"
    ],
    check=True
)

print("FULL 20-EPOCH BASELINE TRAINING COMPLETE")

subprocess.run(
    [
        sys.executable,
        "/kaggle/working/mars_work/code/baseline_analysis.py",
        "--root", "/kaggle/working/mars_work"
    ],
    check=True
)


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 10997.70it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.runn

Resuming baseline from last.pt iteration 12250; config 2e177076344b5638
W&B: connected


baseline stage 2:  10%|▉         | 1224/12250 [02:56<26:56,  6.82it/s, iter=13474, loss=0.030, lr=4.5e-05]

epoch 11/20 | val mIoU4=0.825139 | mIoU3=0.926901 | val loss=0.096041 | best epoch=7


baseline stage 2:  20%|█▉        | 2449/12250 [06:08<24:08,  6.77it/s, iter=14699, loss=0.073, lr=3.8e-05]  

epoch 12/20 | val mIoU4=0.826739 | mIoU3=0.928767 | val loss=0.098156 | best epoch=7


... (15 outputs left out)

epoch 20/20 | val mIoU4=0.833481 | mIoU3=0.934953 | val loss=0.104370 | best epoch=16


baseline stage 2: 100%|██████████| 12250/12250 [31:46<00:00,  6.42it/s, iter=24500, loss=0.038, lr=5e-06]
wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       log/best_checkpoint_event ▁▁█▁██▁▁▁▁
wandb:                   log/elapsed_s ▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
wandb:                       log/epoch ▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇███
wandb:                   log/grad_norm █▅▆▃▆▅▅▅▅▇▂▆▂▃▄▃▂▃▂▂▃█▂▂▁▃▂▃▂▂▅▂▃▆▂▅▆▄▄▄
wandb:                   log/iteration ▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
wandb:                        log/loss ▄▇██▅▃▃▄▃▄▅▅▄▅▅▃▃▄▃▃▃▂▃▂▃▂▂▃▃▃▂▂▁▂▂▂▂▂▁▂
wandb:                          log/lr ████▇▇▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
wandb:   log/nonfinite_grad_norm_total ▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆███
wandb:         log/skipped_steps_total ▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆█
wandb: log/validation_epoch_duration_s ▁██▇██▆▇▇▇
wandb:                             +20

STAGE 2 COMPLETE | 24500/24500 iterations | best val mIoU4=0.838503
STOP: freeze best checkpoint before the one-time final test evaluation.
FULL 20-EPOCH BASELINE TRAINING COMPLETE
Analysis complete (20 validation epochs; W&B not used). Output: /kaggle/working/mars_work/results/baseline/training_analysis


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mars_work/code/baseline_analysis.py', '--root', '/kaggle/working/mars_work'], returncode=0)

In [37]:

from pathlib import Path

W = Path("/kaggle/working/mars_work")

# Confirm training files exist
for name in ("last.pt", "best.pt"):
    p = W / "checkpoints/baseline" / name
    assert p.is_file(), f"Missing checkpoint: {p}"
    print(name, "PRESENT", f"{p.stat().st_size:,} bytes")

assert (W / "results/baseline/validation_log.csv").is_file()
assert (W / "results/baseline/training_analysis").is_dir()

# Reuse the previously verified backup function
backup = save_baseline_backup(
    "backup_baseline_training_complete_pretest"
)

print("\nTRAINING BACKUP CREATED AND VERIFIED:", backup)


last.pt PRESENT 44,856,459 bytes
best.pt PRESENT 44,870,411 bytes
VERIFIED: /kaggle/working/backup_baseline_training_complete_pretest.zip
Files: 113 | Size: 514,984,708 bytes

TRAINING BACKUP CREATED AND VERIFIED: /kaggle/working/backup_baseline_training_complete_pretest.zip


In [38]:

from pathlib import Path
import sys, json, csv, hashlib, shutil, os
import torch

W = Path("/kaggle/working/mars_work")
CK = W / "checkpoints/baseline"
RES = W / "results/baseline"

sys.path.insert(0, str(W / "code"))
import baseline_train as bt

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. Verify completed training and locked configuration
cfg = bt.build_config()
cfg_hash = bt.cu.config_hash(cfg)

last = torch.load(
    CK / "last.pt",
    map_location="cpu",
    weights_only=False
)
best = torch.load(
    CK / "best.pt",
    map_location="cpu",
    weights_only=False
)

assert int(last["iteration"]) == 24500
assert last["config_hash"] == cfg_hash
assert best["config_hash"] == cfg_hash

# 2. Verify the selected best checkpoint
pointer = json.loads(
    (CK / "best_pointer.json").read_text()
)

with open(RES / "validation_log.csv", newline="") as f:
    rows = list(csv.DictReader(f))

assert len(rows) == 20

winner = min(
    rows,
    key=lambda r: (
        -float(r["miou4"]),
        float(r["val_loss"]),
        int(float(r["epoch"]))
    )
)

epoch = int(float(winner["epoch"]))
iteration = int(float(winner["iteration"]))
miou4 = float(winner["miou4"])

assert epoch == 16
assert int(pointer["epoch"]) == epoch
assert int(pointer["iteration"]) == iteration
assert int(best["iteration"]) == iteration
assert abs(float(pointer["miou4"]) - miou4) < 1e-8
assert abs(float(best["best_metric"]) - miou4) < 1e-8
assert int(last["extra"]["best_epoch"]) == epoch

# 3. Verify best.pt against its immutable source
original = CK / pointer["filename"]
assert original.is_file()

source_hash = sha256_file(CK / "best.pt")
assert sha256_file(original) == source_hash

# 4. Create an independent frozen checkpoint
frozen = CK / f"frozen_best_epoch{epoch:02d}.pt"

if frozen.exists():
    assert sha256_file(frozen) == source_hash, (
        "Existing frozen checkpoint differs!"
    )
else:
    temporary = frozen.with_suffix(".pt.tmp")
    assert not temporary.exists(), (
        "Temporary frozen file exists; inspect before retrying."
    )
    shutil.copyfile(CK / "best.pt", temporary)
    assert sha256_file(temporary) == source_hash
    os.replace(temporary, frozen)

assert sha256_file(frozen) == source_hash

# 5. Record the frozen model provenance
record = {
    "status": "FROZEN",
    "selected_epoch": epoch,
    "selected_iteration": iteration,
    "selection_rule": (
        "highest validation mIoU4; "
        "tie: lowest validation loss; "
        "then earliest epoch"
    ),
    "validation_miou4": miou4,
    "validation_miou3": float(winner["miou3"]),
    "validation_loss": float(winner["val_loss"]),
    "frozen_checkpoint": str(frozen.relative_to(W)),
    "source_checkpoint": str(original.relative_to(W)),
    "checkpoint_sha256": source_hash,
    "config_hash": cfg_hash,
    "config": cfg,
    "total_training_iterations": int(last["iteration"]),
    "frozen_split_sha256": sha256_file(
        W / "splits/split_v3_FROZEN.json"
    ),
    "test_evaluation_performed": False
}

record_path = CK / "baseline_frozen.json"

if record_path.exists():
    existing = json.loads(record_path.read_text())
    assert existing == record, (
        "Existing frozen record differs; do not overwrite."
    )
else:
    bt.atomic_json(record_path, record)

print("=" * 55)
print("BASELINE MODEL FROZEN SUCCESSFULLY")
print("=" * 55)
print("Selected epoch:", epoch)
print("Selected iteration:", iteration)
print("Validation mIoU4:", f"{miou4:.6f}")
print("Configuration hash:", cfg_hash)
print("Checkpoint SHA256:", source_hash)
print("Frozen checkpoint:", frozen)
print("Metadata:", record_path)
print("No training or test evaluation performed.")


BASELINE MODEL FROZEN SUCCESSFULLY
Selected epoch: 16
Selected iteration: 19600
Validation mIoU4: 0.838503
Configuration hash: 2e177076344b5638
Checkpoint SHA256: afb9b155e07c617ab69cfa29455beda9318605b0afcec81e07f558a48bcbbda6
Frozen checkpoint: /kaggle/working/mars_work/checkpoints/baseline/frozen_best_epoch16.pt
Metadata: /kaggle/working/mars_work/checkpoints/baseline/baseline_frozen.json
No training or test evaluation performed.


In [39]:

from pathlib import Path

W = Path("/kaggle/working/mars_work")

assert (W / "checkpoints/baseline/frozen_best_epoch16.pt").is_file()
assert (W / "checkpoints/baseline/baseline_frozen.json").is_file()

backup_path = Path(
    "/kaggle/working/backup_baseline_frozen_pretest.zip"
)

if backup_path.exists():
    print("Backup already exists:", backup_path)
else:
    backup_path = save_baseline_backup(
        "backup_baseline_frozen_pretest"
    )

print("POST-FREEZE BACKUP:", backup_path)
print("Size (MB):", round(backup_path.stat().st_size / 1e6, 2))


VERIFIED: /kaggle/working/backup_baseline_frozen_pretest.zip
Files: 115 | Size: 556,409,028 bytes
POST-FREEZE BACKUP: /kaggle/working/backup_baseline_frozen_pretest.zip
Size (MB): 556.41


In [41]:
from pathlib import Path
import json

W = Path("/kaggle/working/mars_work")

print("BASELINE CHECKPOINTS")
for p in sorted((W / "checkpoints/baseline").glob("*")):
    if p.is_file():
        print(p.name, f"{p.stat().st_size:,} bytes")

print("\nFREEZE METADATA")
for name in ("baseline_frozen.json",):
    p = W / "checkpoints/baseline" / name
    if p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2))
    else:
        print(name, "NOT FOUND")

print("\nAVAILABLE EVALUATION SCRIPTS")
for p in sorted((W / "code").glob("*.py")):
    if any(x in p.name.lower() for x in ("eval", "test", "baseline")):
        print(p.name)

BASELINE CHECKPOINTS
baseline_frozen.json 1,656 bytes
best.pt 44,870,411 bytes
best_epoch01_it1225.pt 44,869,579 bytes
best_epoch02_it2450.pt 44,869,579 bytes
best_epoch04_it4900.pt 44,869,579 bytes
best_epoch07_it8575.pt 44,869,579 bytes
best_epoch13_it15925.pt 44,870,411 bytes
best_epoch15_it18375.pt 44,870,411 bytes
best_epoch16_it19600.pt 44,870,411 bytes
best_pointer.json 153 bytes
frozen_best_epoch16.pt 44,870,411 bytes
last.pt 44,856,459 bytes

FREEZE METADATA
{
  "status": "FROZEN",
  "selected_epoch": 16,
  "selected_iteration": 19600,
  "selection_rule": "highest validation mIoU4; tie: lowest validation loss; then earliest epoch",
  "validation_miou4": 0.8385030829983156,
  "validation_miou3": 0.9353173442452696,
  "validation_loss": 0.10268750834969391,
  "frozen_checkpoint": "checkpoints/baseline/frozen_best_epoch16.pt",
  "source_checkpoint": "checkpoints/baseline/best_epoch16_it19600.pt",
  "checkpoint_sha256": "afb9b155e07c617ab69cfa29455beda9318605b0afcec81e07f558a48bcb

In [45]:
%%writefile /kaggle/working/baseline_final_eval_cell.py
from pathlib import Path
from datetime import datetime, timezone
import sys, json, hashlib, time, math
import numpy as np
import torch
from torch.utils.data import DataLoader

W = Path('/kaggle/working/mars_work')
sys.path.insert(0, str(W / 'code'))
import baseline_train as bt
import mars_common as mc

CK = W / 'checkpoints/baseline'
R = W / 'results/baseline'
FROZEN = CK / 'frozen_best_epoch16.pt'
INFO = CK / 'baseline_frozen.json'
PROGRESS = R / 'final_test_progress.json'
FINAL = R / 'final_test_results.json'


def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def json_safe(x):
    if isinstance(x, dict):
        return {str(k): json_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, np.ndarray)):
        return [json_safe(v) for v in x]
    if isinstance(x, np.generic):
        return json_safe(x.item())
    if isinstance(x, float):
        return x if math.isfinite(x) else None
    return x


# Never rerun completed final evaluations or overwrite the record.
if FINAL.exists():
    raise RuntimeError(f'Final test results already exist: {FINAL}. Do NOT evaluate again.')

cfg = bt.build_config()
metadata = json.loads(INFO.read_text())
expected_hash = bt.cu.config_hash(cfg)
assert metadata['status'] == 'FROZEN'
assert metadata['selected_epoch'] == 16
assert metadata['selected_iteration'] == 19600
assert metadata['config_hash'] == expected_hash
assert metadata['config'] == cfg
assert metadata['test_evaluation_performed'] is False
assert sha256(FROZEN) == metadata['checkpoint_sha256'], 'Frozen file SHA256 mismatch'
assert sha256(W / 'splits/split_v3_FROZEN.json') == metadata['frozen_split_sha256']
assert Path(mc.DATA).resolve() == Path('/tmp/mars_data').resolve()

split_sizes = {'test_msl': 311, 'test_mer': 195}
for split, size in split_sizes.items():
    assert (Path(mc.DATA) / f'{split}.csv').is_file(), f'{split} CSV unavailable'
    assert len(mc.MarsSeg(split)) == size, f'{split} size differs from frozen protocol'

if PROGRESS.exists():
    journal = json.loads(PROGRESS.read_text())
    assert journal['checkpoint_sha256'] == metadata['checkpoint_sha256']
    assert journal['config_hash'] == expected_hash
    for split in split_sizes:
        assert split in journal['splits']
else:
    journal = {
        'checkpoint_sha256': metadata['checkpoint_sha256'],
        'config_hash': expected_hash,
        'selected_epoch': 16,
        'splits': {split: {'status': 'pending'} for split in split_sizes},
    }
    bt.atomic_json(PROGRESS, journal)

# IMPORTANT: this initialization retrieves architecture, then replaces ALL
# model parameters with the approved frozen epoch-16 checkpoint.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model = mc.new_model()
state = torch.load(FROZEN, map_location='cpu', weights_only=False)
assert state['config_hash'] == expected_hash
assert int(state['iteration']) == 19600
assert int(state['epoch']) == 16
assert abs(float(state['best_metric']) - metadata['validation_miou4']) < 1e-8
model.load_state_dict(state['model'], strict=True)
model.eval()
print('Frozen epoch-16 model loaded; no training or tuning will occur.')

for split, size in split_sizes.items():
    entry = journal['splits'][split]
    if entry['status'] == 'complete':
        print(f'{split}: already evaluated, reusing saved results (not rerunning).')
        continue
    if entry['status'] != 'pending':
        raise RuntimeError(
            f'{split} was previously started but not completed. '
            f'Inspect {PROGRESS} before deciding whether to rerun it.'
        )

    entry['status'] = 'in_progress'
    entry['started_utc'] = datetime.now(timezone.utc).isoformat()
    bt.atomic_json(PROGRESS, journal)

    print(f'FINAL evaluation: {split} ({size} images)')
    loader = DataLoader(
        mc.MarsSeg(split, train=False), batch_size=cfg['val_batch_size'],
        shuffle=False, num_workers=cfg['workers'], pin_memory=True
    )
    tic = time.monotonic()
    result = mc.evaluate(model, loader)
    elapsed = time.monotonic() - tic

    entry['status'] = 'complete'
    entry['completed_utc'] = datetime.now(timezone.utc).isoformat()
    entry['image_count'] = size
    entry['duration_seconds'] = round(elapsed, 3)
    entry['metrics'] = json_safe(result)
    bt.atomic_json(PROGRESS, journal)
    print(f"{split}: mIoU4={result['miou4']:.6f} | "
          f"mIoU3={result['miou3']:.6f} | pixel accuracy={result['pixacc']:.6f}")
    for name, iou, prec, recall in zip(mc.CLASS_NAMES, result["iou"], result["precision"], result["recall"]):
        print(f"  {name}: IoU={iou:.6f} precision={prec:.6f} recall={recall:.6f}")

assert all(journal['splits'][s]['status'] == 'complete' for s in split_sizes)
report = {
    'status': 'COMPLETE',
    'protocol': 'one-time frozen SegFormer MiT-B0 baseline evaluation',
    'checkpoint_sha256': metadata['checkpoint_sha256'],
    'config_hash': expected_hash,
    'selected_epoch': metadata['selected_epoch'],
    'selected_iteration': metadata['selected_iteration'],
    'class_names': list(mc.CLASS_NAMES),
    'ignore_index': 255,
    'splits': journal['splits'],
}
assert not FINAL.exists(), 'Refusing to overwrite final evaluation results'
bt.atomic_json(FINAL, report)
print('FINAL TEST EVALUATION COMPLETE')
print('Results saved:', FINAL)
print('Progress journal:', PROGRESS)
print('No training was performed. Do not rerun these test evaluations.')


Overwriting /kaggle/working/baseline_final_eval_cell.py


In [46]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "/kaggle/working/baseline_final_eval_cell.py"
    ],
    check=True
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 8320.74it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.classifier.bias 

Frozen epoch-16 model loaded; no training or tuning will occur.
FINAL evaluation: test_msl (311 images)
test_msl: mIoU4=0.676215 | mIoU3=0.864864 | pixel accuracy=0.927891
  soil: IoU=0.921961 precision=0.975177 recall=0.944118
  bedrock: IoU=0.812220 precision=0.854477 recall=0.942607
  sand: IoU=0.860411 precision=0.974741 recall=0.880033
  big rock: IoU=0.110267 precision=0.126809 recall=0.458070
FINAL evaluation: test_mer (195 images)
test_mer: mIoU4=0.700983 | mIoU3=0.788847 | pixel accuracy=0.882720
  soil: IoU=0.809490 precision=0.873022 recall=0.917515
  bedrock: IoU=0.767651 precision=0.843188 recall=0.895496
  sand: IoU=0.789400 precision=0.954527 recall=0.820246
  big rock: IoU=0.437390 precision=0.670309 recall=0.557277
FINAL TEST EVALUATION COMPLETE
Results saved: /kaggle/working/mars_work/results/baseline/final_test_results.json
Progress journal: /kaggle/working/mars_work/results/baseline/final_test_progress.json
No training was performed. Do not rerun these test evaluati

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/baseline_final_eval_cell.py'], returncode=0)

In [47]:

from pathlib import Path
import json
import shutil
import hashlib

W = Path("/kaggle/working/mars_work")
R = W / "results/baseline"
CK = W / "checkpoints/baseline"

# Verify final results
report = json.loads((R / "final_test_results.json").read_text())
frozen = json.loads((CK / "baseline_frozen.json").read_text())

assert report["status"] == "COMPLETE"
assert report["checkpoint_sha256"] == frozen["checkpoint_sha256"]
assert report["selected_epoch"] == 16

for split, expected in [("test_msl", 311), ("test_mer", 195)]:
    entry = report["splits"][split]
    assert entry["status"] == "complete"
    assert entry["image_count"] == expected
    assert "miou4" in entry["metrics"]
    print(split, "VERIFIED")

# Preserve the exact evaluation script for reproducibility
source = Path("/kaggle/working/baseline_final_eval_cell.py")
destination = W / "code/baseline_final_eval.py"

assert source.is_file(), "Original evaluation script missing"

if destination.exists():
    assert destination.read_bytes() == source.read_bytes()
else:
    shutil.copy2(source, destination)

# Create the final archive using your verified backup function
backup = Path("/kaggle/working/backup_baseline_final_evaluated.zip")
assert not backup.exists(), "Backup exists; do not overwrite it"

backup = save_baseline_backup("backup_baseline_final_evaluated")

print("\nFINAL BASELINE BACKUP VERIFIED:", backup)
print("Size (MB):", round(backup.stat().st_size / 1e6, 2))


test_msl VERIFIED
test_mer VERIFIED
VERIFIED: /kaggle/working/backup_baseline_final_evaluated.zip
Files: 118 | Size: 556,414,297 bytes

FINAL BASELINE BACKUP VERIFIED: /kaggle/working/backup_baseline_final_evaluated.zip
Size (MB): 556.41


In [48]:

from pathlib import Path
import zipfile, hashlib, json

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/mars_segformer_trained_model.zip")

files = [
    "checkpoints/baseline/frozen_best_epoch16.pt",
    "checkpoints/baseline/baseline_frozen.json",
    "configs/baseline_protocol.json",
    "code/mars_common.py",
    "code/ckpt_utils.py",
    "code/mars_det.py",
    "code/tracking.py",
    "code/recovery.py",
    "code/baseline_train.py",
]

for name in files:
    assert (W / name).is_file(), f"Missing: {name}"

# Confirm model matches the frozen checkpoint record
metadata = json.loads(
    (W / files[1]).read_text()
)
model = W / files[0]
model_hash = hashlib.sha256(model.read_bytes()).hexdigest()

assert model_hash == metadata["checkpoint_sha256"]
assert metadata["selected_epoch"] == 16

assert not OUT.exists(), "Share ZIP already exists"

with zipfile.ZipFile(
    OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=1
) as z:
    for name in files:
        z.write(W / name, arcname=name)

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None

print("MODEL SHARING PACKAGE CREATED")
print("File:", OUT)
print("Size:", round(OUT.stat().st_size / 1e6, 2), "MB")
print("Model SHA256:", model_hash)
print("Files included:", len(files))


MODEL SHARING PACKAGE CREATED
File: /kaggle/working/mars_segformer_trained_model.zip
Size: 41.44 MB
Model SHA256: afb9b155e07c617ab69cfa29455beda9318605b0afcec81e07f558a48bcbbda6
Files included: 9


WHATEVER IS BELOW THIS IS OLD NOTEBOOK